In [ ]:
# CELDA 1 - Cargar el dataset (selecciona el archivo dataset_solar_popayan.csv cuando aparezca el botón)
from google.colab import files
subido = files.upload()
print("Archivos cargados:", list(subido.keys()))

In [ ]:
# =============================================================================
# CELDA 1B - INSTALACIÓN Y CONFIGURACIÓN DEL STACK NEURONAL E IA GENERATIVA
# =============================================================================
import sys, subprocess, warnings, os
from google.colab import userdata

def _asegurar_paquetes_neuronales():
    """Instala solo los paquetes que no estén presentes en el entorno actual."""
    import importlib.util
    requeridos = {
        "torch": "torch>=2.2.0",
        "transformers": "transformers>=4.40.0",
        "sentence_transformers": "sentence-transformers>=2.7.0",
        "accelerate": "accelerate",
        "peft": "peft",
        "datasets": "datasets",
        "einops": "einops",
        "tqdm": "tqdm",
        "captum": "captum",
        "umap": "umap-learn",
        "faiss": "faiss-cpu"
    }

    faltan = []
    for mod, pipname in requeridos.items():
        spec = importlib.util.find_spec(mod)
        if spec is None:
            # Manejo de nombres especiales de importación vs pip
            if mod == "umap" and importlib.util.find_spec("umap") is not None:
                continue
            if mod == "faiss" and importlib.util.find_spec("faiss") is not None:
                continue
            faltan.append(pipname)

    if faltan:
        print("Instalando paquetes neuronales faltantes:", faltan)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + faltan, check=False)
    else:
        print("Todos los paquetes neuronales ya están instalados.")

_asegurar_paquetes_neuronales()

import torch
import random
import numpy as np

# --- Configuración de Hardware e hilos ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 80)
print(f"DISPOSITIVO NEURONAL DETECTADO: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria Reservada: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
else:
    print("AVISO: Corriendo en CPU. Los entrenamientos de redes y LLM usarán fallbacks ligeros.")
print("=" * 80)

# --- Configuración estricta de reproducibilidad ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True, warn_only=True)
print("Semillas de reproducibilidad de PyTorch y algoritmos deterministas configurados con éxito.")

Instalando paquetes neuronales faltantes: ['captum', 'faiss-cpu']
DISPOSITIVO NEURONAL DETECTADO: cpu
AVISO: Corriendo en CPU. Los entrenamientos de redes y LLM usarán fallbacks ligeros.
Semillas de reproducibilidad de PyTorch y algoritmos deterministas configurados con éxito.


In [ ]:
# =============================================================================
# CELDA 3B - CLASIFICADOR NEURONAL MULTIMODAL (HogarAdopcionNet)
# =============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

# --- 1. Definición de la Red Neuronal Multimodular HogarAdopcionNet ---
class HogarAdopcionNet(nn.Module):
    def __init__(self, num_numeric_features, cat_cardinalities, embed_dims, temporal_dim=24):
        super(HogarAdopcionNet, self).__init__()

        # A. Rama Numérica (Tabular Continua)
        self.numeric_branch = nn.Sequential(
            nn.Linear(num_numeric_features, 64),
            nn.BatchNorm1d(64),
            nn.Mish(),
            nn.Dropout(0.25),
            nn.Linear(64, 32),
            nn.Mish()
        )

        # B. Rama Categórica (Embeddings dedicados)
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, dim) for num_classes, dim in zip(cat_cardinalities, embed_dims)
        ])
        total_embed_dim = sum(embed_dims)
        self.categorical_branch = nn.Sequential(
            nn.Linear(total_embed_dim, 32),
            nn.Mish(),
            nn.Dropout(0.20)
        ) if total_embed_dim > 0 else nn.Identity()

        # C. Rama Temporal (Perfil horario de 24 horas)
        self.temporal_branch = nn.Sequential(
            nn.Linear(temporal_dim, 32),
            nn.BatchNorm1d(32),
            nn.Mish(),
            nn.Dropout(0.20),
            nn.Linear(32, 16),
            nn.Mish()
        )

        # Fusión Tardía y Clasificador Final
        fusion_dim = 32 + (32 if total_embed_dim > 0 else 0) + 16
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 32),
            nn.Mish(),
            nn.Dropout(0.20),
            nn.Linear(32, 3), # 3 clases de salida: No, Tal vez, Sí
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x_num, x_cat, x_temp):
        # Procesamiento de cada rama
        out_num = self.numeric_branch(x_num)

        if len(self.embeddings) > 0:
            embeds = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            out_cat = torch.cat(embeds, dim=1)
            out_cat = self.categorical_branch(out_cat)
        else:
            out_cat = None

        out_temp = self.temporal_branch(x_temp)

        # Fusión tardía de representaciones latentes
        if out_cat is not None:
            fused = torch.cat([out_num, out_cat, out_temp], dim=1)
        else:
            fused = torch.cat([out_num, out_temp], dim=1)

        return self.classifier(fused)

# --- 2. Dataset Custom de PyTorch para Carga Eficiente ---
class MultimodalDataset(Dataset):
    def __init__(self, df_features, num_cols, cat_cols, temp_cols, targets=None):
        self.x_num = torch.tensor(df_features[num_cols].values, dtype=torch.float32)
        self.x_cat = torch.tensor(df_features[cat_cols].values, dtype=torch.long)
        self.x_temp = torch.tensor(df_features[temp_cols].values, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.long) if targets is not None else None

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        if self.targets is not None:
            return self.x_num[idx], self.x_cat[idx], self.x_temp[idx], self.targets[idx]
        return self.x_num[idx], self.x_cat[idx], self.x_temp[idx]

print("Estructura del clasificador multimodal HogarAdopcionNet y su Dataset inicializados correctamente.")

Estructura del clasificador multimodal HogarAdopcionNet y su Dataset inicializados correctamente.


In [ ]:
# =============================================================================
# CELDA 3C - ENTRENAMIENTO Y EVALUACIÓN DE HOGARADOPCIONNET
# =============================================================================
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# --- 1. Preparación de Variables Reales del Entorno (Defensivo Autónomo) ---
try:
    # Intentar usar las variables si ya existen en el espacio de nombres
    num_cols_m = NUM_CLS.copy()
    cat_cols_m = CAT_CLS.copy()
    temp_cols_m = HORAS.copy()
    X_tr_m = X_tr.copy()
    X_te_m = X_te.copy()
    y_tr_m = y_tr.copy()
    y_te_m = y_te.copy()
except NameError:
    # Reconstrucción autónoma si el pipeline global monolítico no se ha ejecutado aún o se reinició el kernel
    print("Variables globales no detectadas en memoria. Reconstruyendo entorno de forma autónoma...")

    NUM_CLS = ["estrato", "num_personas", "num_menores", "num_adultos_mayores", "ingreso_mensual_cop", "consumo_kwh_mes", "tarifa_kwh_cop",
               "area_vivienda_m2", "area_cubierta_m2", "num_paneles_estimados", "potencia_panel_w", "porcentaje_ahorro", "retorno_simple_anios",
               "carga_familiar", "ratio_consumo_ingreso_pct", "kwh_por_persona", "kwh_por_m2", "indice_eficiencia_vivienda",
               "cobertura_solar_pct", "brecha_digital", "kwp_calc"]
    CAT_CLS = ["zona", "nivel_educativo", "ocupacion_principal", "tiene_empresa", "tipo_cubierta", "sombras_presentes", "tiene_internet",
               "tiene_smartphone", "tiene_medidor_inteligente", "recibe_subsidio", "conoce_reciclaje_paneles", "interes_comunidad_energetica"]
    HORAS = [f"h{h:02d}" for h in range(24)]

    num_cols_m = NUM_CLS.copy()
    cat_cols_m = CAT_CLS.copy()
    temp_cols_m = HORAS.copy()

    # Función defensiva para localizar el dataset en el entorno de Colab
    def localizar_csv_fallback():
        nombres = ["dataset_solar_popayan*.csv"]
        carpetas = [".", "/content", "/content/sample_data", "/mnt/user-data/outputs", "/mnt/user-data/uploads"]
        for c in carpetas:
            for n in nombres:
                r = sorted(glob.glob(os.path.join(c, n)))
                if r:
                    return r[-1]
        return None

    ruta_csv_segura = localizar_csv_fallback()
    if ruta_csv_segura is not None:
        print(f"Cargando dataset detectado desde: {ruta_csv_segura}")
        df_local = pd.read_csv(ruta_csv_segura, encoding="utf-8")
    else:
        print("AVISO: Dataset no encontrado físicamente en Colab. Generando set sintético equivalente de autocuración...")
        # Generar un dataset sintético idéntico para que el pipeline funcione autónomamente
        n_samples = 200
        rg = np.random.default_rng(SEED)
        data = {
            "estrato": rg.choice([1, 2, 3], size=n_samples),
            "num_personas": rg.integers(1, 8, size=n_samples),
            "num_menores": rg.integers(0, 4, size=n_samples),
            "num_adultos_mayores": rg.integers(0, 2, size=n_samples),
            "ingreso_mensual_cop": rg.uniform(800000, 5000000, size=n_samples),
            "consumo_kwh_mes": rg.uniform(80, 450, size=n_samples),
            "tarifa_kwh_cop": rg.uniform(700, 950, size=n_samples),
            "area_vivienda_m2": rg.uniform(40, 250, size=n_samples),
            "area_cubierta_m2": rg.uniform(10, 100, size=n_samples),
            "num_paneles_estimados": rg.integers(3, 12, size=n_samples),
            "potencia_panel_w": rg.choice([400, 450, 550], size=n_samples),
            "porcentaje_ahorro": rg.uniform(30, 80, size=n_samples),
            "retorno_simple_anios": rg.uniform(3, 20, size=n_samples),
            "zona": rg.choice(["Urbana", "Rural"], size=n_samples),
            "nivel_educativo": rg.choice(["Primaria", "Secundaria", "Técnico", "Universitario"], size=n_samples),
            "ocupacion_principal": rg.choice(["Empleado", "Independiente", "Hogar", "Pensionado"], size=n_samples),
            "tiene_empresa": rg.choice(["Sí", "No"], size=n_samples),
            "tipo_cubierta": rg.choice(["Teja de barro", "Zinc", "Plancha"], size=n_samples),
            "sombras_presentes": rg.choice(["Sí", "No"], size=n_samples),
            "tiene_internet": rg.choice(["Sí", "No"], size=n_samples),
            "tiene_smartphone": rg.choice(["Sí", "No"], size=n_samples),
            "tiene_medidor_inteligente": rg.choice(["Sí", "No"], size=n_samples),
            "recibe_subsidio": rg.choice(["Sí", "No"], size=n_samples),
            "conoce_reciclaje_paneles": rg.choice(["Sí", "No"], size=n_samples),
            "interes_comunidad_energetica": rg.choice(["Sí", "No"], size=n_samples),
            "dispuesto_instalar_paneles": rg.choice(["Sí", "No", "Tal vez"], size=n_samples)
        }
        for h in range(24):
            data[f"h{h:02d}"] = rg.uniform(0.1, 1.5, size=n_samples)
        df_local = pd.DataFrame(data)

    # Ingeniería de características mínima necesaria para HogarAdopcionNet
    df_local["kwp_calc"] = df_local["num_paneles_estimados"] * df_local["potencia_panel_w"] / 1000.0
    df_local["carga_familiar"] = (df_local["num_menores"] + df_local["num_adultos_mayores"]) / df_local["num_personas"]
    df_local["ratio_consumo_ingreso_pct"] = 100.0 * df_local["consumo_kwh_mes"] * df_local["tarifa_kwh_cop"] / df_local["ingreso_mensual_cop"]
    df_local["kwh_por_persona"] = df_local["consumo_kwh_mes"] / df_local["num_personas"]
    df_local["kwh_por_m2"] = df_local["consumo_kwh_mes"] / df_local["area_vivienda_m2"]

    # Índices referenciales para eficiencias
    REF_PP = df_local.groupby("estrato")["kwh_por_persona"].median().to_dict()
    REF_M2 = df_local.groupby("estrato")["kwh_por_m2"].median().to_dict()
    rpp = df_local["estrato"].map(REF_PP)
    rm2 = df_local["estrato"].map(REF_M2)
    df_local["indice_eficiencia_vivienda"] = (100.0 * (1 - 0.5 * (df_local["kwh_por_persona"] / (2 * rpp) + df_local["kwh_por_m2"] / (2 * rm2)))).clip(0, 100)
    df_local["cobertura_solar_pct"] = 100.0 * df_local["kwp_calc"] * 4.25 * 30 * 0.78 / df_local["consumo_kwh_mes"]
    df_local["brecha_digital"] = (df_local["tiene_internet"] == "No").astype(int) + (df_local["tiene_smartphone"] == "No").astype(int)

    # Particionamiento (Se incluyen las variables de perfiles horarios 'temp_cols_m' para evitar el KeyError)
    X_all_local = df_local[num_cols_m + cat_cols_m + temp_cols_m].copy()
    y_all_local = df_local["dispuesto_instalar_paneles"].map({"No": 0, "Tal vez": 1, "Sí": 2}).astype(int).values

    X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_all_local, y_all_local, test_size=0.20, stratify=y_all_local, random_state=SEED)

# Asegurar de forma defensiva la existencia de la variable de directorios
if 'DIRS' not in globals():
    OUT_FALLBACK = "/content/outputs" if os.path.exists("/content") else "outputs"
    DIRS = {"modelos": f"{OUT_FALLBACK}/modelos"}

# Label Encoders para cada columna categórica para pasarlas a embeddings
cat_encoders = {}
cat_cardinalities = []
embed_dims = []

# Ajustar encoders y transformar columnas categóricas a enteros de forma segura
for col in cat_cols_m:
    le = LabelEncoder()
    combined_series = pd.concat([X_tr_m[col].astype(str), X_te_m[col].astype(str)])
    le.fit(combined_series)
    cat_encoders[col] = le

    X_tr_m[col] = le.transform(X_tr_m[col].astype(str))
    X_te_m[col] = le.transform(X_te_m[col].astype(str))

    cardinality = len(le.classes_)
    cat_cardinalities.append(cardinality)
    embed_dims.append(max(2, min(50, (cardinality + 1) // 2)))

# --- 2. Instanciación de Datasets y DataLoaders ---
train_dataset = MultimodalDataset(X_tr_m, num_cols_m, cat_cols_m, temp_cols_m, y_tr_m)
test_dataset = MultimodalDataset(X_te_m, num_cols_m, cat_cols_m, temp_cols_m, y_te_m)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- 3. Inicialización Defensiva de HogarAdopcionNet ---
model = HogarAdopcionNet(
    num_numeric_features=len(num_cols_m),
    cat_cardinalities=cat_cardinalities,
    embed_dims=embed_dims,
    temporal_dim=len(temp_cols_m)
).to(DEVICE)

criterion = nn.NLLLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Asegurar que el directorio para guardar modelos exista
os.makedirs(DIRS["modelos"], exist_ok=True)

# --- 4. Bucle de Entrenamiento con Early Stopping Frugal ---
epochs = 15
best_loss = float('inf')
patience = 3
patience_counter = 0

print(f"Iniciando entrenamiento en: {DEVICE}")
for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for x_num, x_cat, x_temp, y in train_loader:
        x_num, x_cat, x_temp, y = x_num.to(DEVICE), x_cat.to(DEVICE), x_temp.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        log_probs = model(x_num, x_cat, x_temp)
        loss = criterion(log_probs, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x_num.size(0)

    train_loss /= len(train_loader.dataset)

    # Validación rápida de pérdida
    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for x_num, x_cat, x_temp, y in test_loader:
            x_num, x_cat, x_temp, y = x_num.to(DEVICE), x_cat.to(DEVICE), x_temp.to(DEVICE), y.to(DEVICE)
            log_probs = model(x_num, x_cat, x_temp)
            loss = criterion(log_probs, y)
            val_loss += loss.item() * x_num.size(0)

            preds = log_probs.argmax(dim=1)
            correct += (preds == y).sum().item()

    val_loss /= len(test_loader.dataset)
    val_acc = correct / len(test_loader.dataset)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Época {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}")

    # Early stopping defensivo
    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f"{DIRS['modelos']}/hogar_adopcion_net_best.pth")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping activado en la época {epoch}.")
            break

# Cargar mejores pesos
model.load_state_dict(torch.load(f"{DIRS['modelos']}/hogar_adopcion_net_best.pth" ))
print("Entrenamiento del clasificador multimodal completado y mejores pesos restaurados con éxito.")

Variables globales no detectadas en memoria. Reconstruyendo entorno de forma autónoma...
Cargando dataset detectado desde: ./dataset_solar_popayan.csv
Iniciando entrenamiento en: cpu
Época 01/15 | Train Loss: 1.0100 | Val Loss: 0.9770 | Val Acc: 54.50%
Época 05/15 | Train Loss: 0.9055 | Val Loss: 0.9029 | Val Acc: 59.00%
Época 10/15 | Train Loss: 0.8653 | Val Loss: 0.8804 | Val Acc: 59.00%
Early stopping activado en la época 13.
Entrenamiento del clasificador multimodal completado y mejores pesos restaurados con éxito.


In [ ]:
# =============================================================================
# CELDA 2 - PROYECTO INTEGRADOR (Diplomado Energía Circular Inteligente 5.0 e IA)
# "Autogeneración Solar Fotovoltaica en Hogares de Popayán, Cauca (2026-2030)"
# Pipeline completo: ETL -> EDA -> ML supervisado -> ML no supervisado ->
# series temporales -> optimización -> economía circular -> dashboard Gradio -> exportación
# Ejecutar DESPUÉS de la celda 1 (carga de dataset_solar_popayan.csv).
# =============================================================================
import os, sys, io, re, glob, json, math, time, shutil, zipfile, subprocess, warnings, itertools, pickle
warnings.filterwarnings("ignore")
T0 = time.time()

def _asegurar_paquetes():
    """Instala solo lo que falte (equivale a !pip install de gradio, plotly, shap, statsmodels, prophet,
    scikit-learn, xgboost, lightgbm, pulp, kaleido). Se hace con subprocess para funcionar dentro de una sola celda."""
    import importlib
    requeridos = {"gradio": "gradio", "plotly": "plotly", "shap": "shap", "statsmodels": "statsmodels",
                  "prophet": "prophet", "sklearn": "scikit-learn", "xgboost": "xgboost",
                  "lightgbm": "lightgbm", "pulp": "pulp", "kaleido": "kaleido", "scipy": "scipy", "joblib": "joblib"}
    faltan = []
    for mod, pipname in requeridos.items():
        try:
            m_ = importlib.import_module(mod)
            # PuLP 4.x cambió por completo su API; se fija <4 (el código además trae respaldo con scipy.optimize.milp)
            if mod == "pulp" and int(str(m_.__version__).split(".")[0]) >= 4:
                faltan.append("pulp<4")
        except Exception:
            faltan.append("pulp<4" if mod == "pulp" else pipname)
    if faltan:
        print("Instalando paquetes faltantes:", faltan)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + faltan, check=False)
_asegurar_paquetes()

import numpy as np
import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import linkage
import gradio as gr

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold, cross_validate,
                                     cross_val_predict, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor, StackingClassifier, IsolationForest)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, roc_curve,
                             average_precision_score, precision_recall_curve, matthews_corrcoef, cohen_kappa_score,
                             log_loss, confusion_matrix, r2_score, mean_absolute_error, mean_squared_error,
                             silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score,
                             precision_score, recall_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
import shap
import pulp
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
try:
    from prophet import Prophet
    PROPHET_OK = True
except Exception as _e:
    PROPHET_OK = False
    print("AVISO: Prophet no disponible (", _e, "). Se continuará con SARIMAX y línea base estacional.")

import logging
for _n in ("prophet", "cmdstanpy", "pystan"):
    logging.getLogger(_n).setLevel(logging.ERROR)
import logging
for _lg in ("cmdstanpy", "prophet", "prophet.plot", "matplotlib", "numexpr"):
    logging.getLogger(_lg).setLevel(logging.ERROR)
SEED = 42
np.random.seed(SEED)
pio.templates.default = "plotly_white"
LAUNCH = os.environ.get("NO_LAUNCH", "0") != "1"

IN_COLAB = "google.colab" in sys.modules
OUT = "/content/outputs" if IN_COLAB else "outputs"
DIRS = {"base": OUT, "graficos": f"{OUT}/graficos", "tablas": f"{OUT}/tablas", "modelos": f"{OUT}/modelos"}
for _d in DIRS.values():
    os.makedirs(_d, exist_ok=True)

print("=" * 100)
print("AVISO ÉTICO Y DE DATOS: el dataset utilizado es SINTÉTICO, generado y calibrado con parámetros de referencia")
print("de Popayán (UPME-IDEAM, DANE). No contiene información personal de personas reales. Aun así, cualquier uso con")
print("datos reales de hogares debe cumplir la Ley 1581 de 2012 (Habeas Data) y el Decreto 1377 de 2013:")
print("consentimiento informado, finalidad explícita, minimización de datos y anonimización.")
print("=" * 100)

# ------------------------------------------------------------------ utilidades de registro
FIGS, TABLAS, MODELOS = {}, {}, {}
_PNG_OK = {"estado": None}

def reg_fig(nombre, fig, clave=False):
    """Registra una figura Plotly, la exporta a HTML (siempre) y a PNG (solo claves, si kaleido funciona)."""
    FIGS[nombre] = fig
    try:
        fig.write_html(f"{DIRS['graficos']}/{nombre}.html", include_plotlyjs="cdn")
    except Exception as e:
        print("  (no se pudo exportar HTML de", nombre, ":", e, ")")
    if clave and _PNG_OK["estado"] is not False:
        try:
            fig.write_image(f"{DIRS['graficos']}/{nombre}.png", width=1100, height=650, scale=1.5)
            _PNG_OK["estado"] = True
        except Exception as e:
            ok = False
            if _PNG_OK["estado"] is None:           # primer fallo: intentar instalar Chrome para kaleido (plotly>=6)
                try:
                    pio.get_chrome()
                    fig.write_image(f"{DIRS['graficos']}/{nombre}.png", width=1100, height=650, scale=1.5)
                    ok = True
                except Exception:
                    ok = False
            _PNG_OK["estado"] = ok
            if not ok:
                print("  (PNG no disponible en este entorno; se exportan los gráficos en HTML interactivo)")
    return fig

def reg_tabla(nombre, df_, index=False):
    TABLAS[nombre] = df_
    df_.to_csv(f"{DIRS['tablas']}/{nombre}.csv", index=index, encoding="utf-8-sig")
    return df_

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}")

# =============================================================================
# BLOQUE 1. ETL Y FEATURE ENGINEERING
# =============================================================================
log("BLOQUE 1: ETL y Feature Engineering")

def localizar_csv():
    nombres = ["dataset_solar_popayan*.csv"]
    carpetas = [".", "/content", "/content/sample_data", "/mnt/user-data/outputs", "/mnt/user-data/uploads"]
    for c in carpetas:
        for n in nombres:
            r = sorted(glob.glob(os.path.join(c, n)))
            if r:
                return r[-1]
    raise FileNotFoundError("No se encontró dataset_solar_popayan.csv. Ejecuta primero la celda 1 (files.upload()).")

RUTA_CSV = localizar_csv()
df_raw = pd.read_csv(RUTA_CSV, encoding="utf-8")
print(f"Dataset cargado: {RUTA_CSV} -> {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")

HORAS = [f"h{h:02d}" for h in range(24)]
COLS_ESPERADAS = ["id_hogar", "estrato", "zona", "consumo_kwh_mes", "dispuesto_instalar_paneles", "ahorro_anual_cop"] + HORAS
_faltan = [c for c in COLS_ESPERADAS if c not in df_raw.columns]
if _faltan:
    raise ValueError(f"Faltan columnas esperadas en el CSV: {_faltan}")

# ---- 1.1 Calidad de datos
def tabla_calidad(d):
    filas = []
    def chk(regla, ok_mask):
        filas.append([regla, int((~ok_mask).sum()), f"{100 * ok_mask.mean():.1f}%"])
    chk("Sin valores nulos (por fila)", ~d.isna().any(axis=1))
    chk("id_hogar único", ~d.duplicated("id_hogar", keep=False))
    chk("Sin filas duplicadas", ~d.duplicated(keep=False))
    chk("Suma h00-h23 = consumo diario", np.isclose(d[HORAS].sum(axis=1), d["consumo_kwh_dia_promedio"], atol=1e-3))
    chk("Factura = consumo x tarifa", (d["consumo_kwh_mes"] * d["tarifa_kwh_cop"] - d["costo_factura_mes_cop"]).abs() <= 1)
    chk("Ahorro mensual <= factura", d["ahorro_mensual_cop"] <= d["costo_factura_mes_cop"])
    chk("Área requerida <= área cubierta", d["area_requerida_m2"] <= d["area_cubierta_m2"])
    chk("Consumo anual = 12 x mensual", (d["consumo_anual_kwh"] - 12 * d["consumo_kwh_mes"]).abs() < 0.2)
    chk("Menores + mayores < personas", (d["num_menores"] + d["num_adultos_mayores"]) < d["num_personas"])
    chk("Estrato 3 sin subsidio", ~((d["estrato"] == 3) & (d["recibe_subsidio"] == "Sí")))
    chk("Sin internet ni smartphone => IA 'Ninguno'",
        ~((d["tiene_internet"] == "No") & (d["tiene_smartphone"] == "No") & (d["dispositivo_ia_interes"] != "Ninguno")))
    chk("VPN>0 coherente con TIR>10%", (d["vpn_10_anios_cop"] > 0) == (d["tir_porcentaje"] > 10))
    return pd.DataFrame(filas, columns=["regla_de_coherencia", "filas_que_incumplen", "cumplimiento"])

calidad = reg_tabla("calidad_datos", tabla_calidad(df_raw))
print(calidad.to_string(index=False))

# ---- 1.2 Winsorización (1%-99%) de variables con colas largas. Se conservan las originales en df_raw.
COLS_WINSOR = ["ingreso_mensual_cop", "demanda_maxima_kw", "inversion_estimada_cop", "retorno_simple_meses",
               "retorno_simple_anios", "vpn_10_anios_cop", "tir_porcentaje"]
df = df_raw.copy()
resumen_w = []
for c in COLS_WINSOR:
    lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
    n_mod = int(((df[c] < lo) | (df[c] > hi)).sum())
    df[c] = df[c].clip(lo, hi)
    resumen_w.append([c, round(lo, 3), round(hi, 3), n_mod])
reg_tabla("winsorizacion", pd.DataFrame(resumen_w, columns=["variable", "p01", "p99", "valores_ajustados"]))
print("Winsorización 1%-99% aplicada a:", COLS_WINSOR)

# ---- 1.3 Ingeniería de características
HSP_REF, PR_REF = 4.25, 0.78
_tmp = df.assign(_pp=df["consumo_kwh_mes"] / df["num_personas"], _m2=df["consumo_kwh_mes"] / df["area_vivienda_m2"])
REF_PP = _tmp.groupby("estrato")["_pp"].median().to_dict()
REF_M2 = _tmp.groupby("estrato")["_m2"].median().to_dict()

def ingenieria_caracteristicas(d):
    """Variables derivadas calculables también desde el formulario del simulador."""
    d = d.copy()
    d["kwp_calc"] = d["num_paneles_estimados"] * d["potencia_panel_w"] / 1000.0
    d["carga_familiar"] = (d["num_menores"] + d["num_adultos_mayores"]) / d["num_personas"]
    d["ratio_consumo_ingreso_pct"] = 100.0 * d["consumo_kwh_mes"] * d["tarifa_kwh_cop"] / d["ingreso_mensual_cop"]
    d["kwh_por_persona"] = d["consumo_kwh_mes"] / d["num_personas"]
    d["kwh_por_m2"] = d["consumo_kwh_mes"] / d["area_vivienda_m2"]
    rpp = d["estrato"].map(REF_PP); rm2 = d["estrato"].map(REF_M2)
    d["indice_eficiencia_vivienda"] = (100.0 * (1 - 0.5 * (d["kwh_por_persona"] / (2 * rpp) + d["kwh_por_m2"] / (2 * rm2)))).clip(0, 100)
    d["cobertura_solar_pct"] = 100.0 * d["kwp_calc"] * HSP_REF * 30 * PR_REF / d["consumo_kwh_mes"]
    d["brecha_digital"] = (d["tiene_internet"] == "No").astype(int) + (d["tiene_smartphone"] == "No").astype(int)
    return d

def agregar_perfil(d):
    """Variables del perfil horario (solo disponibles en el dataset, no en los formularios)."""
    d = d.copy()
    tot = d[HORAS].sum(axis=1)
    d["ratio_diurno_6_17"] = d[[f"h{h:02d}" for h in range(6, 18)]].sum(axis=1) / tot
    d["ratio_manana_6_8"] = d[["h06", "h07", "h08"]].sum(axis=1) / tot
    d["ratio_mediodia_12_14"] = d[["h12", "h13", "h14"]].sum(axis=1) / tot
    d["ratio_noche_18_22"] = d[[f"h{h:02d}" for h in range(18, 23)]].sum(axis=1) / tot
    d["ratio_madrugada_0_5"] = d[[f"h{h:02d}" for h in range(0, 6)]].sum(axis=1) / tot
    d["factor_pico_horario"] = d[HORAS].max(axis=1) / d[HORAS].mean(axis=1)
    return d

df = agregar_perfil(ingenieria_caracteristicas(df))
df["objetivo_adopcion"] = df["dispuesto_instalar_paneles"]
CLASES = ["No", "Tal vez", "Sí"]
print("Características creadas: carga_familiar, ratio_consumo_ingreso_pct, indice_eficiencia_vivienda, cobertura_solar_pct,",
      "brecha_digital, ratios del perfil horario (diurno, mañana, mediodía, noche, madrugada), factor_pico_horario")

# ---- 1.4 Diccionario de datos (descripciones + estadísticas automáticas)
DESC = {
 "id_hogar": "Identificador único del hogar", "comuna": "Comuna (1-9) o corregimiento rural", "barrio": "Barrio o vereda (sintético)",
 "estrato": "Estrato socioeconómico (1-3)", "tipo_vivienda": "Casa / Apartamento / Vivienda rural", "zona": "Urbana o Rural",
 "num_personas": "Personas en el hogar", "num_menores": "Menores de edad", "num_adultos_mayores": "Adultos mayores",
 "ingreso_mensual_cop": "Ingreso mensual del hogar (COP)", "jefe_hogar_genero": "Género del jefe de hogar", "nivel_educativo": "Nivel educativo del jefe",
 "ocupacion_principal": "Ocupación principal", "tiene_empresa": "Pequeño emprendimiento en el hogar",
 "area_vivienda_m2": "Área de la vivienda (m²)", "area_cubierta_m2": "Área de cubierta disponible (m²)", "tipo_cubierta": "Material de la cubierta",
 "inclinacion_cubierta_grados": "Inclinación de la cubierta (°)", "orientacion_cubierta": "Orientación de la cubierta", "sombras_presentes": "Sombras sobre la cubierta",
 "sombras_descripcion": "Origen de las sombras", "consumo_kwh_mes": "Consumo eléctrico mensual (kWh)", "consumo_kwh_dia_promedio": "Consumo diario promedio (kWh)",
 "demanda_maxima_kw": "Demanda máxima estimada (kW)", "factor_carga": "Factor de carga (potencia media / máxima)", "costo_factura_mes_cop": "Factura mensual (COP)",
 "tarifa_kwh_cop": "Tarifa efectiva (COP/kWh)", "consumo_anual_kwh": "Consumo anual (kWh)", "variacion_estacional_pct": "Variación estacional lluvioso vs seco (%)",
 "mes_mayor_consumo": "Mes de mayor consumo", "mes_menor_consumo": "Mes de menor consumo", "tiene_medidor_inteligente": "Medidor inteligente",
 "recibe_subsidio": "Recibe subsidio", "radiacion_promedio_kwh_m2_dia": "Radiación solar (kWh/m²/día)", "horas_sol_pico": "Horas sol pico (h/día)",
 "generacion_potencial_kwp": "Potencia FV dimensionada (kWp)", "num_paneles_estimados": "Número de paneles", "potencia_panel_w": "Potencia por panel (W)",
 "area_requerida_m2": "Área requerida por paneles (m²)", "generacion_diaria_estimada_kwh": "Generación diaria estimada (kWh)",
 "generacion_mensual_estimada_kwh": "Generación mensual estimada (kWh)", "inversion_estimada_cop": "Inversión estimada (COP)", "costo_kwp_cop": "Costo por kWp (COP)",
 "ahorro_mensual_cop": "Ahorro mensual (COP)", "ahorro_anual_cop": "Ahorro anual (COP)", "porcentaje_ahorro": "Ahorro como % de la factura",
 "retorno_simple_meses": "Retorno simple (meses)", "retorno_simple_anios": "Retorno simple (años)", "vpn_10_anios_cop": "VPN a 10 años (COP)",
 "tir_porcentaje": "TIR a 10 años (%)", "requiere_financiacion": "Requiere financiación", "fuente_financiacion": "Fuente de financiación",
 "dispositivo_ia_interes": "Interés en dispositivo/servicio de IA", "nivel_apropiacion_tecnologica": "Apropiación tecnológica", "tiene_internet": "Tiene internet",
 "tiene_smartphone": "Tiene smartphone", "interes_comunidad_energetica": "Interés en comunidad energética", "conoce_reciclaje_paneles": "Conoce el reciclaje de paneles",
 "dispuesto_instalar_paneles": "OBJETIVO de clasificación: Sí / Tal vez / No", "principal_barrera": "Principal barrera de adopción",
 "principal_motivacion": "Principal motivación", "disposicion_pago_mensual_cop": "Disposición de pago mensual (COP)", "prioridad_gobierno": "Prioridad solicitada al gobierno",
 "num_neveras": "Neveras", "potencia_nevera_w": "Potencia nevera (W)", "num_televisores": "Televisores", "potencia_tv_w": "Potencia TV (W)",
 "num_lavadoras": "Lavadoras", "potencia_lavadora_w": "Potencia lavadora (W)", "num_ventiladores": "Ventiladores", "potencia_ventilador_w": "Potencia ventilador (W)",
 "num_bombillas_led": "Bombillas LED", "potencia_bombilla_w": "Potencia bombilla (W)", "num_computadores": "Computadores", "num_equipos_especiales": "Equipos especiales",
 "carga_familiar": "[FE] (menores + adultos mayores) / personas", "ratio_consumo_ingreso_pct": "[FE] Factura / ingreso (%)", "kwh_por_persona": "[FE] kWh/mes por persona",
 "kwh_por_m2": "[FE] kWh/mes por m²", "indice_eficiencia_vivienda": "[FE] Índice 0-100 de eficiencia vs. mediana del estrato (mayor = más eficiente)",
 "cobertura_solar_pct": "[FE] Generación nominal / consumo (%)", "brecha_digital": "[FE] 0-2: ausencia de internet y smartphone", "kwp_calc": "[FE] kWp = paneles x W / 1000",
 "ratio_diurno_6_17": "[FE] Fracción del consumo entre 06:00 y 17:59", "ratio_manana_6_8": "[FE] Fracción 06-08 h", "ratio_mediodia_12_14": "[FE] Fracción 12-14 h",
 "ratio_noche_18_22": "[FE] Fracción 18-22 h", "ratio_madrugada_0_5": "[FE] Fracción 00-05 h", "factor_pico_horario": "[FE] Pico horario / media horaria",
 "objetivo_adopcion": "Copia del objetivo (dispuesto_instalar_paneles)"}
for h in HORAS:
    DESC[h] = f"Consumo medio en la hora {h[1:]}:00 (kWh)"

def construir_diccionario(d):
    filas = []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            tipo = "numérica"; rango = f"{s.min():.4g} a {s.max():.4g}"
        else:
            tipo = "categórica"; u = sorted(s.astype(str).unique())
            rango = " | ".join(u) if len(u) <= 8 else f"{len(u)} categorías"
        filas.append([c, tipo, DESC.get(c, ""), rango, int(s.isna().sum()), int(s.nunique())])
    return pd.DataFrame(filas, columns=["variable", "tipo", "descripcion", "rango_o_categorias", "nulos", "valores_unicos"])

diccionario = reg_tabla("diccionario_datos", construir_diccionario(df))
reg_tabla("dataset_modelado", df)

# =============================================================================
# BLOQUE 2. EDA
# =============================================================================
log("BLOQUE 2: EDA")
COLOR_E = {1: "#c0392b", 2: "#e6a817", 3: "#2471a3"}
COLOR_T = {"No": "#c0392b", "Tal vez": "#e6a817", "Sí": "#1e8449"}
NUM_EDA = ["num_personas", "ingreso_mensual_cop", "area_vivienda_m2", "area_cubierta_m2", "consumo_kwh_mes", "demanda_maxima_kw",
           "factor_carga", "costo_factura_mes_cop", "generacion_potencial_kwp", "inversion_estimada_cop", "ahorro_anual_cop",
           "porcentaje_ahorro", "retorno_simple_anios", "vpn_10_anios_cop", "tir_porcentaje", "carga_familiar",
           "ratio_consumo_ingreso_pct", "indice_eficiencia_vivienda", "cobertura_solar_pct", "ratio_diurno_6_17",
           "ratio_noche_18_22", "disposicion_pago_mensual_cop"]
CAT_EDA = ["estrato", "zona", "tipo_vivienda", "nivel_educativo", "ocupacion_principal", "tipo_cubierta", "orientacion_cubierta",
           "sombras_presentes", "tiene_internet", "tiene_smartphone", "nivel_apropiacion_tecnologica", "dispositivo_ia_interes",
           "principal_barrera", "principal_motivacion", "prioridad_gobierno", "fuente_financiacion", "dispuesto_instalar_paneles"]

desc_num = df[NUM_EDA].describe(percentiles=[.25, .5, .75]).T
desc_num["asimetria"] = df[NUM_EDA].skew(); desc_num["curtosis"] = df[NUM_EDA].kurt()
reg_tabla("estadisticas_descriptivas_modelo", desc_num.round(3), index=True)
reg_tabla("resumen_por_estrato", df.groupby("estrato")[["consumo_kwh_mes", "ingreso_mensual_cop", "area_cubierta_m2", "generacion_potencial_kwp",
                                                          "inversion_estimada_cop", "ahorro_anual_cop", "retorno_simple_anios"]].mean().round(2), index=True)

# ---- 2.1 Univariado
def fig_univariado(var):
    if var in NUM_EDA:
        fig = make_subplots(rows=1, cols=2, column_widths=[0.7, 0.3], subplot_titles=("Distribución", "Boxplot por estrato"))
        fig.add_trace(go.Histogram(x=df[var], nbinsx=40, marker_color="#2471a3", name=var), row=1, col=1)
        for s in (1, 2, 3):
            fig.add_trace(go.Box(y=df.loc[df.estrato == s, var], name=f"E{s}", marker_color=COLOR_E[s], showlegend=False), row=1, col=2)
        fig.update_layout(title=f"{var}  (media={df[var].mean():.2f} | mediana={df[var].median():.2f} | asimetría={df[var].skew():.2f})", height=470)
    else:
        vc = df[var].astype(str).value_counts().reset_index(); vc.columns = [var, "hogares"]
        fig = px.bar(vc, x=var, y="hogares", text="hogares", color=var, title=f"Frecuencias de {var}")
        fig.update_layout(showlegend=False, height=470)
    return fig

for v in ["consumo_kwh_mes", "ingreso_mensual_cop", "retorno_simple_anios", "dispuesto_instalar_paneles"]:
    reg_fig(f"eda_univariado_{v}", fig_univariado(v), clave=(v == "consumo_kwh_mes"))

# ---- 2.2 Bivariado
def fig_bivariado(x, y, color):
    col = None if color == "(ninguno)" else color
    num_x, num_y = x in NUM_EDA, y in NUM_EDA
    if num_x and num_y:
        try:
            fig = px.scatter(df, x=x, y=y, color=col if col is None else df[col].astype(str), trendline="ols", opacity=0.65,
                             title=f"{y} vs {x}")
        except Exception:
            fig = px.scatter(df, x=x, y=y, color=col if col is None else df[col].astype(str), opacity=0.65, title=f"{y} vs {x}")
        r, p = stats.pearsonr(df[x], df[y]); rho, _ = stats.spearmanr(df[x], df[y])
        fig.update_layout(title=f"{y} vs {x}  |  Pearson r={r:.2f}  Spearman ρ={rho:.2f}")
    elif (not num_x) and num_y:
        fig = px.box(df, x=x, y=y, color=x, points="outliers", title=f"{y} según {x}")
        grupos = [g[y].values for _, g in df.groupby(x)]
        if len(grupos) > 1:
            h, p = stats.kruskal(*grupos)
            fig.update_layout(title=f"{y} según {x}  |  Kruskal-Wallis H={h:.1f}, p={p:.2g}")
    elif num_x and (not num_y):
        fig = px.box(df, x=y, y=x, color=y, points="outliers", title=f"{x} según {y}")
    else:
        ct = pd.crosstab(df[x], df[y], normalize="index") * 100
        fig = px.imshow(ct.round(1), text_auto=True, aspect="auto", color_continuous_scale="Blues", title=f"% fila: {y} dentro de cada {x}")
        chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df[x], df[y]))
        fig.update_layout(title=f"{y} dentro de cada {x}  |  χ²={chi2:.1f}, p={p:.2g}")
    fig.update_layout(height=520)
    return fig

reg_fig("eda_bivariado_consumo_ingreso", fig_bivariado("ingreso_mensual_cop", "consumo_kwh_mes", "estrato"), clave=True)
reg_fig("eda_bivariado_inversion_ahorro", fig_bivariado("inversion_estimada_cop", "ahorro_anual_cop", "estrato"))

# ---- 2.3 Multivariado: PCA 2D/3D, t-SNE, dendrograma
EDA_MV = [c for c in NUM_EDA if c not in ("disposicion_pago_mensual_cop",)]
X_mv = StandardScaler().fit_transform(df[EDA_MV])
pca_mv = PCA(n_components=min(10, X_mv.shape[1]), random_state=SEED).fit(X_mv)
Z = pca_mv.transform(X_mv)
var_exp = pca_mv.explained_variance_ratio_
fig_var = go.Figure()
fig_var.add_bar(x=[f"PC{i + 1}" for i in range(len(var_exp))], y=var_exp * 100, name="Varianza explicada")
fig_var.add_scatter(x=[f"PC{i + 1}" for i in range(len(var_exp))], y=np.cumsum(var_exp) * 100, name="Acumulada", mode="lines+markers")
fig_var.update_layout(title="PCA: varianza explicada (%)", yaxis_title="%", height=420)
reg_fig("pca_varianza", fig_var)
cargas = pd.DataFrame(pca_mv.components_[:3].T, index=EDA_MV, columns=["PC1", "PC2", "PC3"]).round(3)
reg_tabla("pca_cargas", cargas, index=True)

dfZ = pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1], "PC3": Z[:, 2], "estrato": df["estrato"].astype(str),
                    "adopcion": df["dispuesto_instalar_paneles"], "zona": df["zona"]})
fig_pca2 = px.scatter(dfZ, x="PC1", y="PC2", color="adopcion", symbol="estrato", color_discrete_map=COLOR_T, opacity=0.75,
                      title=f"PCA 2D (PC1 {var_exp[0]*100:.1f}% + PC2 {var_exp[1]*100:.1f}%)")
fig_pca2.update_layout(height=520)
fig_pca3 = px.scatter_3d(dfZ, x="PC1", y="PC2", z="PC3", color="adopcion", symbol="estrato", color_discrete_map=COLOR_T, opacity=0.75,
                         title=f"PCA 3D (varianza acumulada {var_exp[:3].sum()*100:.1f}%)")
fig_pca3.update_layout(height=620)
reg_fig("pca_2d", fig_pca2, clave=True); reg_fig("pca_3d", fig_pca3)

tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED)
Zt = tsne.fit_transform(X_mv)
dfT = pd.DataFrame({"tSNE1": Zt[:, 0], "tSNE2": Zt[:, 1], "estrato": df["estrato"].astype(str), "adopcion": df["dispuesto_instalar_paneles"]})
fig_tsne = px.scatter(dfT, x="tSNE1", y="tSNE2", color="adopcion", symbol="estrato", color_discrete_map=COLOR_T, opacity=0.75,
                      title="t-SNE (perplexity=30) coloreado por disposición de adopción")
fig_tsne.update_layout(height=520)
reg_fig("tsne_2d", fig_tsne, clave=True)

_rng = np.random.default_rng(SEED)
idx_dend = np.sort(_rng.choice(len(df), 90, replace=False))
fig_dend0 = ff.create_dendrogram(X_mv[idx_dend], labels=[f"H{df.id_hogar.iloc[i][1:]}-E{df.estrato.iloc[i]}" for i in idx_dend],
                                 linkagefun=lambda x: linkage(x, "ward"))
fig_dend0.update_layout(title="Dendrograma (Ward) de 90 hogares muestreados", height=560, xaxis_tickangle=90)
reg_fig("dendrograma_eda", fig_dend0)

# ---- 2.4 Matriz de correlación
corr_cols = [c for c in NUM_EDA]
corr = df[corr_cols].corr(method="spearman").round(2)
fig_corr = px.imshow(corr, text_auto=False, zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto",
                     title="Matriz de correlación de Spearman (variables numéricas clave)")
fig_corr.update_layout(height=780)
reg_fig("matriz_correlacion", fig_corr, clave=True)
reg_tabla("correlaciones_spearman", corr, index=True)

# ---- 2.5 Análisis del target
tab_t = pd.crosstab(df["estrato"], df["dispuesto_instalar_paneles"], normalize="index")[CLASES] * 100
fig_t1 = px.bar(tab_t.reset_index().melt(id_vars="estrato", var_name="respuesta", value_name="porcentaje"), x="estrato", y="porcentaje",
                color="respuesta", color_discrete_map=COLOR_T, barmode="stack", text_auto=".1f", title="Disposición a instalar paneles por estrato (%)")
tab_z = pd.crosstab(df["zona"], df["dispuesto_instalar_paneles"], normalize="index")[CLASES] * 100
fig_t2 = px.bar(tab_z.reset_index().melt(id_vars="zona", var_name="respuesta", value_name="porcentaje"), x="zona", y="porcentaje",
                color="respuesta", color_discrete_map=COLOR_T, barmode="stack", text_auto=".1f", title="Disposición por zona (%)")
fig_t3 = px.box(df, x="dispuesto_instalar_paneles", y="retorno_simple_anios", color="dispuesto_instalar_paneles", color_discrete_map=COLOR_T,
                category_orders={"dispuesto_instalar_paneles": CLASES}, title="Retorno simple (años) según disposición")
fig_target = make_subplots(rows=1, cols=3, subplot_titles=("Por estrato (%)", "Por zona (%)", "Retorno simple (años)"))
for tr in fig_t1.data: fig_target.add_trace(tr, row=1, col=1)
for tr in fig_t2.data:
    tr.showlegend = False; fig_target.add_trace(tr, row=1, col=2)
for tr in fig_t3.data:
    tr.showlegend = False; fig_target.add_trace(tr, row=1, col=3)
fig_target.update_layout(barmode="stack", height=480, title="Análisis del target: dispuesto_instalar_paneles")
reg_fig("analisis_target", fig_target, clave=True)
bal = df["dispuesto_instalar_paneles"].value_counts(normalize=True).reindex(CLASES) * 100
print("Balance del target (%):", bal.round(1).to_dict())

# ---- 2.6 Pareto de barreras
def fig_pareto(poblacion="No + Tal vez", nivel="Barrera"):
    d = df
    if poblacion == "Solo No": d = df[df.dispuesto_instalar_paneles == "No"]
    elif poblacion == "No + Tal vez": d = df[df.dispuesto_instalar_paneles != "Sí"]
    if nivel == "Barrera":
        s = d["principal_barrera"].value_counts()
        ttl = "Pareto de la principal barrera de adopción"
    else:
        s = (d["estrato"].map(lambda e: f"E{e}") + " | " + d["zona"] + " | " + d["principal_barrera"]).value_counts()
        ttl = "Pareto de segmentos (estrato | zona | barrera)"
    s = s.sort_values(ascending=False)
    acum = s.cumsum() / s.sum() * 100
    n20 = max(1, int(math.ceil(0.2 * len(s))))
    pct20 = acum.iloc[n20 - 1]
    n80 = int((acum < 80).sum() + 1)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    colores = ["#c0392b" if i < n20 else "#aab7b8" for i in range(len(s))]
    fig.add_bar(x=s.index, y=s.values, marker_color=colores, name="Hogares", secondary_y=False)
    fig.add_scatter(x=s.index, y=acum.values, mode="lines+markers", name="% acumulado", line=dict(color="#2471a3"), secondary_y=True)
    fig.add_hline(y=80, line_dash="dash", line_color="gray", secondary_y=True)
    fig.update_yaxes(title_text="Hogares", secondary_y=False); fig.update_yaxes(title_text="% acumulado", range=[0, 105], secondary_y=True)
    fig.update_layout(title=f"{ttl} | población: {poblacion} (n={len(d)}). El 20% superior ({n20} de {len(s)} categorías) explica el {pct20:.1f}%; "
                            f"se necesitan {n80} para llegar al 80%", height=520, xaxis_tickangle=-45, title_font_size=13)
    tabla = pd.DataFrame({"categoria": s.index, "hogares": s.values, "pct": (s / s.sum() * 100).round(1).values, "pct_acumulado": acum.round(1).values})
    return fig, tabla

fig_par, tabla_par = fig_pareto("No + Tal vez", "Barrera")
fig_par2, tabla_par2 = fig_pareto("No + Tal vez", "Segmentos")
reg_fig("pareto_barreras", fig_par, clave=True); reg_fig("pareto_segmentos", fig_par2)
reg_tabla("pareto_barreras", tabla_par); reg_tabla("pareto_segmentos", tabla_par2)
print("Pareto - barreras principales:\n", tabla_par.to_string(index=False))
log("EDA completado")

# =============================================================================
# BLOQUE 3. CLASIFICACIÓN: ADOPCIÓN DE PANELES (No / Tal vez / Sí)
# =============================================================================
log("BLOQUE 3: Clasificación de adopción")
# Decisión de diseño anti-fuga de información (data leakage): se EXCLUYEN del modelo las variables que se miden
# después de la decisión o que casi la determinan: disposicion_pago_mensual_cop (=0 para quienes dicen No),
# principal_barrera, principal_motivacion, prioridad_gobierno (actitudes posteriores) y dispositivo_ia_interes.
# Esas variables se analizan en el EDA/Pareto, pero no se usan para predecir.
NUM_CLS = ["estrato", "num_personas", "num_menores", "num_adultos_mayores", "ingreso_mensual_cop", "consumo_kwh_mes", "tarifa_kwh_cop",
           "area_vivienda_m2", "area_cubierta_m2", "num_paneles_estimados", "potencia_panel_w", "porcentaje_ahorro", "retorno_simple_anios",
           "carga_familiar", "ratio_consumo_ingreso_pct", "kwh_por_persona", "kwh_por_m2", "indice_eficiencia_vivienda",
           "cobertura_solar_pct", "brecha_digital", "kwp_calc"]
CAT_CLS = ["zona", "nivel_educativo", "ocupacion_principal", "tiene_empresa", "tipo_cubierta", "sombras_presentes", "tiene_internet",
           "tiene_smartphone", "tiene_medidor_inteligente", "recibe_subsidio", "conoce_reciclaje_paneles", "interes_comunidad_energetica"]
FEATS_CLS = NUM_CLS + CAT_CLS
X_all = df[FEATS_CLS].copy()
y_all = df["dispuesto_instalar_paneles"].map({c: i for i, c in enumerate(CLASES)}).astype(int).values
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED)
idx_te = X_te.index
print(f"Entrenamiento: {len(X_tr)} | Prueba: {len(X_te)} | Variables: {len(FEATS_CLS)} ({len(NUM_CLS)} numéricas, {len(CAT_CLS)} categóricas)")

def nuevo_preproceso(num=NUM_CLS, cat=CAT_CLS):
    return ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)], verbose_feature_names_out=True)

CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ---- Ajuste ligero de hiperparámetros (RandomizedSearch, validación cruzada interna solo con el set de entrenamiento)
def afinar(nombre, modelo, espacio, n_iter=10):
    pipe = Pipeline([("prep", nuevo_preproceso()), ("clf", modelo)])
    rs = RandomizedSearchCV(pipe, espacio, n_iter=n_iter, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                            scoring="f1_macro", random_state=SEED, n_jobs=1, refit=False)
    rs.fit(X_tr, y_tr)
    mejores = {k.replace("clf__", ""): v for k, v in rs.best_params_.items()}
    log(f"  ajuste {nombre}: f1_macro CV3={rs.best_score_:.3f} con {mejores}")
    return mejores

xgb_base = dict(objective="multi:softprob", eval_metric="mlogloss", random_state=SEED, n_jobs=-1, verbosity=0, tree_method="hist")
lgb_base = dict(random_state=SEED, n_jobs=-1, verbose=-1)
p_xgb = afinar("XGBoost", XGBClassifier(**xgb_base),
               {"clf__n_estimators": [150, 250, 400], "clf__max_depth": [2, 3, 4, 5], "clf__learning_rate": [0.03, 0.05, 0.1],
                "clf__subsample": [0.7, 0.85, 1.0], "clf__colsample_bytree": [0.6, 0.8, 1.0], "clf__min_child_weight": [1, 3, 5]})
p_lgb = afinar("LightGBM", LGBMClassifier(**lgb_base),
               {"clf__n_estimators": [150, 250, 400], "clf__num_leaves": [7, 15, 31], "clf__learning_rate": [0.03, 0.05, 0.1],
                "clf__subsample": [0.7, 0.85, 1.0], "clf__colsample_bytree": [0.6, 0.8, 1.0], "clf__min_child_samples": [10, 20, 30],
                "clf__subsample_freq": [1]})
p_rf = afinar("Random Forest", RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced_subsample"),
              {"clf__n_estimators": [300, 500], "clf__max_depth": [None, 6, 10], "clf__min_samples_leaf": [1, 2, 4],
               "clf__max_features": ["sqrt", 0.5]}, n_iter=8)

def modelos_clasificacion():
    lr = LogisticRegression(max_iter=3000, C=1.0, random_state=SEED)
    rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced_subsample", **p_rf)
    xg = XGBClassifier(**{**xgb_base, **p_xgb})
    lg = LGBMClassifier(**{**lgb_base, **p_lgb})
    sv = SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, random_state=SEED)
    ml = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation="relu", alpha=1e-3, learning_rate_init=1e-3,
                       early_stopping=True, validation_fraction=0.15, n_iter_no_change=25, max_iter=1000, random_state=SEED)
    st = StackingClassifier(estimators=[("lr", clone(lr)), ("rf", clone(rf)), ("xgb", clone(xg)), ("lgbm", clone(lg)), ("svm", clone(sv))],
                            final_estimator=LogisticRegression(max_iter=3000, random_state=SEED), cv=5, stack_method="predict_proba", n_jobs=1)
    return {"Regresión Logística": lr, "Random Forest": rf, "XGBoost": xg, "LightGBM": lg, "SVM (RBF)": sv, "MLP (128-64-32)": ml, "Stacking": st}

PIPES = {n: Pipeline([("prep", nuevo_preproceso()), ("clf", m)]) for n, m in modelos_clasificacion().items()}

def brier_multiclase(y, P):
    Y = np.eye(P.shape[1])[y]
    return float(np.mean(np.sum((P - Y) ** 2, axis=1)))

def ece_multiclase(y, P, bins=10):
    conf = P.max(axis=1); pred = P.argmax(axis=1); ok = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def metricas_clf(y, P):
    pred = P.argmax(axis=1); Y = np.eye(P.shape[1])[y]
    return {"accuracy": accuracy_score(y, pred), "balanced_acc": balanced_accuracy_score(y, pred), "f1_macro": f1_score(y, pred, average="macro"),
            "f1_weighted": f1_score(y, pred, average="weighted"), "roc_auc_ovr": roc_auc_score(y, P, multi_class="ovr", average="macro"),
            "pr_auc_macro": float(np.mean([average_precision_score(Y[:, k], P[:, k]) for k in range(P.shape[1])])),
            "mcc": matthews_corrcoef(y, pred), "kappa": cohen_kappa_score(y, pred), "brier": brier_multiclase(y, P),
            "log_loss": log_loss(y, P, labels=[0, 1, 2]), "ece": ece_multiclase(y, P)}

filas_cv, filas_te, PROBA_TE = [], [], {}
scoring_cv = {"accuracy": "accuracy", "f1_macro": "f1_macro", "balanced_acc": "balanced_accuracy", "roc_auc_ovr": "roc_auc_ovr", "neg_log_loss": "neg_log_loss"}
for nombre, pipe in PIPES.items():
    t1 = time.time()
    cvres = cross_validate(pipe, X_tr, y_tr, cv=CV5, scoring=scoring_cv, n_jobs=1, error_score="raise")
    pipe.fit(X_tr, y_tr)
    P = pipe.predict_proba(X_te); PROBA_TE[nombre] = P
    m = metricas_clf(y_te, P)
    filas_cv.append({"modelo": nombre, **{f"cv_{k}": cvres[f"test_{k}"].mean() for k in scoring_cv}, "cv_f1_macro_std": cvres["test_f1_macro"].std()})
    filas_te.append({"modelo": nombre, **m})
    log(f"  {nombre:22s} CV f1_macro={cvres['test_f1_macro'].mean():.3f} | test f1_macro={m['f1_macro']:.3f} AUC={m['roc_auc_ovr']:.3f} ({time.time() - t1:.0f}s)")
tab_cv = pd.DataFrame(filas_cv).set_index("modelo"); tab_te = pd.DataFrame(filas_te).set_index("modelo")
tab_clf = tab_cv.join(tab_te, how="left").round(4).sort_values("cv_f1_macro", ascending=False)
reg_tabla("clasificacion_comparacion_modelos", tab_clf, index=True)
MEJOR = tab_clf.index[0]          # selección por validación cruzada (no por el set de prueba) para evitar optimismo
print("MEJOR MODELO (por f1_macro en CV):", MEJOR)
MEJOR_PIPE = PIPES[MEJOR]; P_best = PROBA_TE[MEJOR]; pred_best = P_best.argmax(axis=1)

# ---- gráficos de comparación
met_plot = ["accuracy", "balanced_acc", "f1_macro", "roc_auc_ovr", "pr_auc_macro", "mcc", "kappa"]
dfp = tab_te[met_plot].reset_index().melt(id_vars="modelo", var_name="métrica", value_name="valor")
fig_cmp = px.bar(dfp, x="modelo", y="valor", color="métrica", barmode="group", title="Comparación de 7 modelos en el set de prueba (20%)")
fig_cmp.update_layout(height=520, xaxis_tickangle=-20)
reg_fig("clf_comparacion_modelos", fig_cmp, clave=True)
fig_cv = go.Figure()
fig_cv.add_bar(x=tab_cv.index, y=tab_cv["cv_f1_macro"], error_y=dict(type="data", array=tab_cv["cv_f1_macro_std"]), marker_color="#2471a3", name="f1_macro CV (5 folds)")
fig_cv.update_layout(title="F1-macro en validación cruzada estratificada (media ± desv.)", height=420, yaxis_range=[0, 1])
reg_fig("clf_cv_f1", fig_cv)

# ---- matriz de confusión
cm = confusion_matrix(y_te, pred_best, labels=[0, 1, 2]); cmn = cm / cm.sum(axis=1, keepdims=True)
fig_cm = make_subplots(rows=1, cols=2, subplot_titles=("Conteos", "Normalizada por fila (recall)"))
fig_cm.add_trace(go.Heatmap(z=cm, x=CLASES, y=CLASES, text=cm, texttemplate="%{text}", colorscale="Blues", showscale=False), row=1, col=1)
fig_cm.add_trace(go.Heatmap(z=cmn, x=CLASES, y=CLASES, text=np.round(cmn, 2), texttemplate="%{text}", colorscale="Greens", showscale=False), row=1, col=2)
fig_cm.update_xaxes(title_text="Predicho"); fig_cm.update_yaxes(title_text="Real", autorange="reversed")
fig_cm.update_layout(title=f"Matriz de confusión - {MEJOR}", height=450)
reg_fig("clf_matriz_confusion", fig_cm, clave=True)

def curvas_roc_pr(y, P, nombre):
    Y = np.eye(3)[y]
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Curvas ROC (uno-contra-resto)", "Curvas Precisión-Recall"))
    for k, c in enumerate(CLASES):
        fpr, tpr, _ = roc_curve(Y[:, k], P[:, k]); auc = roc_auc_score(Y[:, k], P[:, k])
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{c} (AUC={auc:.3f})", line=dict(color=COLOR_T[c])), row=1, col=1)
        pr, rc, _ = precision_recall_curve(Y[:, k], P[:, k]); ap = average_precision_score(Y[:, k], P[:, k])
        fig.add_trace(go.Scatter(x=rc, y=pr, mode="lines", name=f"{c} (AP={ap:.3f})", line=dict(color=COLOR_T[c], dash="dot")), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=1, col=1)
    fig.update_xaxes(title_text="FPR", row=1, col=1); fig.update_yaxes(title_text="TPR", row=1, col=1)
    fig.update_xaxes(title_text="Recall", row=1, col=2); fig.update_yaxes(title_text="Precisión", row=1, col=2)
    fig.update_layout(title=f"{nombre}: ROC macro AUC={roc_auc_score(y, P, multi_class='ovr'):.3f}", height=470)
    return fig
reg_fig("clf_roc_pr", curvas_roc_pr(y_te, P_best, MEJOR), clave=True)

# ---- calibración isotónica (comparación antes/después)
cal_iso = CalibratedClassifierCV(estimator=clone(MEJOR_PIPE), method="isotonic", cv=5)
cal_iso.fit(X_tr, y_tr)
P_cal = cal_iso.predict_proba(X_te)
m_ant = metricas_clf(y_te, P_best); m_des = metricas_clf(y_te, P_cal)
tab_cal = pd.DataFrame({"antes (sin calibrar)": m_ant, "después (isotónica)": m_des}).loc[["brier", "log_loss", "ece", "accuracy", "f1_macro", "roc_auc_ovr"]].round(4)
reg_tabla("clasificacion_calibracion", tab_cal, index=True)
print(tab_cal.to_string())
USAR_CAL = m_des["brier"] < m_ant["brier"]
MODELO_FINAL_CLF = cal_iso if USAR_CAL else MEJOR_PIPE
print("Modelo final para simulador:", MEJOR + (" + calibración isotónica" if USAR_CAL else " (sin calibrar; la calibración no mejora el Brier)"))

fig_cal = make_subplots(rows=1, cols=3, subplot_titles=[f"Clase '{c}' (uno-contra-resto)" for c in CLASES], shared_yaxes=True)
Yte = np.eye(3)[y_te]
for k, c in enumerate(CLASES):
    for P_, nm, col in ((P_best, "Sin calibrar", "#c0392b"), (P_cal, "Isotónica", "#1e8449")):
        try:
            pt, pp = calibration_curve(Yte[:, k], P_[:, k], n_bins=6, strategy="quantile")
            fig_cal.add_trace(go.Scatter(x=pp, y=pt, mode="lines+markers", name=nm, line=dict(color=col), showlegend=(k == 0)), row=1, col=k + 1)
        except Exception:
            pass
    fig_cal.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=1, col=k + 1)
    fig_cal.update_xaxes(title_text="Probabilidad predicha", row=1, col=k + 1)
fig_cal.update_yaxes(title_text="Frecuencia observada", row=1, col=1)
fig_cal.update_layout(title=f"Curvas de calibración - {MEJOR}  |  Brier: {m_ant['brier']:.3f} -> {m_des['brier']:.3f}", height=460)
reg_fig("clf_calibracion", fig_cal, clave=True)

# ---- interpretabilidad: Permutation Importance (modelo final, variables originales)
pi = permutation_importance(MODELO_FINAL_CLF, X_te, y_te, scoring="f1_macro", n_repeats=15, random_state=SEED, n_jobs=1)
tab_pi = pd.DataFrame({"variable": FEATS_CLS, "importancia": pi.importances_mean, "desv": pi.importances_std}).sort_values("importancia", ascending=False)
reg_tabla("clasificacion_permutation_importance", tab_pi.round(4))
top_pi = tab_pi.head(15).iloc[::-1]
fig_pi = go.Figure(go.Bar(x=top_pi["importancia"], y=top_pi["variable"], orientation="h", error_x=dict(type="data", array=top_pi["desv"]), marker_color="#2471a3"))
fig_pi.update_layout(title="Permutation Importance (caída del F1-macro al permutar, set de prueba)", height=520)
reg_fig("clf_permutation_importance", fig_pi, clave=True)

# ---- SHAP (sobre el mejor modelo basado en árboles; los explicadores exactos requieren árboles)
arboles = [n for n in ("Random Forest", "XGBoost", "LightGBM") if n in tab_cv.index]
ARBOL = max(arboles, key=lambda n: tab_cv.loc[n, "cv_f1_macro"])
arbol_pipe = PIPES[ARBOL]
prep_a = arbol_pipe.named_steps["prep"]
Xt_te = prep_a.transform(X_te); nombres_ohe = [re.sub(r"^(num|cat)__", "", n) for n in prep_a.get_feature_names_out()]
def _orig(n):
    for c in CAT_CLS:
        if n.startswith(c + "_"): return c
    return n
orig_de = np.array([_orig(n) for n in nombres_ohe])
expl = shap.TreeExplainer(arbol_pipe.named_steps["clf"])
sv_raw = expl.shap_values(Xt_te)
def _a3d(sv, n, f, k=3):
    if isinstance(sv, list): a = np.stack(sv, axis=2)
    else:
        a = np.asarray(sv)
        if a.ndim == 2: a = a[:, :, None]
        if a.shape[0] == k and a.shape[1] == n: a = np.transpose(a, (1, 2, 0))
    return a
SV = _a3d(sv_raw, Xt_te.shape[0], Xt_te.shape[1])
base_vals = np.atleast_1d(expl.expected_value)
# agregación a variables originales (suma de dummies por fila: tratamiento correcto de variables categóricas)
ORIG_VARS = list(dict.fromkeys(orig_de))
SV_O = np.stack([SV[:, orig_de == v, :].sum(axis=1) for v in ORIG_VARS], axis=1)         # (n, F, k)
imp_cls = np.abs(SV_O).mean(axis=0)                                                          # (F, k)
tab_shap = pd.DataFrame(imp_cls, index=ORIG_VARS, columns=[f"SHAP|{c}|" for c in CLASES]); tab_shap["SHAP_medio_abs_total"] = imp_cls.sum(axis=1)
tab_shap = tab_shap.sort_values("SHAP_medio_abs_total", ascending=False)
reg_tabla("clasificacion_shap_importancia", tab_shap.round(4), index=True)
top = tab_shap.head(15).iloc[::-1]
fig_shap_bar = go.Figure()
for k, c in enumerate(CLASES):
    fig_shap_bar.add_bar(y=top.index, x=top[f"SHAP|{c}|"], orientation="h", name=c, marker_color=COLOR_T[c])
fig_shap_bar.update_layout(barmode="stack", title=f"SHAP - importancia media absoluta por clase ({ARBOL})", height=560)
reg_fig("clf_shap_resumen", fig_shap_bar, clave=True)

def fig_shap_beeswarm(clase="Sí", top_n=12):
    k = CLASES.index(clase)
    orden = np.argsort(-np.abs(SV_O[:, :, k]).mean(axis=0))[:top_n][::-1]
    fig = go.Figure(); rng_ = np.random.default_rng(SEED)
    for pos, j in enumerate(orden):
        v = ORIG_VARS[j]; col = X_te[v]
        val = col.astype("category").cat.codes.astype(float).values if not pd.api.types.is_numeric_dtype(col) else col.values.astype(float)
        rng_v = (val - val.min()) / (np.ptp(val) + 1e-9)
        fig.add_trace(go.Scatter(x=SV_O[:, j, k], y=pos + rng_.uniform(-0.28, 0.28, len(val)), mode="markers",
                                 marker=dict(size=5, color=rng_v, colorscale="RdBu_r", showscale=(pos == len(orden) - 1),
                                             colorbar=dict(title="valor<br>(bajo→alto)")), text=[f"{v}={x}" for x in col.values], hoverinfo="text+x", showlegend=False))
    fig.update_yaxes(tickvals=list(range(len(orden))), ticktext=[ORIG_VARS[j] for j in orden])
    fig.update_layout(title=f"SHAP beeswarm - impacto sobre la probabilidad de '{clase}' ({ARBOL})", xaxis_title="Valor SHAP", height=560)
    return fig
reg_fig("clf_shap_beeswarm_si", fig_shap_beeswarm("Sí"))

def fig_shap_force(i=0, clase="Sí", top_n=10):
    """Gráfico tipo 'force/waterfall' para un hogar del set de prueba."""
    k = CLASES.index(clase); i = int(np.clip(i, 0, SV_O.shape[0] - 1))
    ph = SV_O[i, :, k]; orden = np.argsort(-np.abs(ph))[:top_n][::-1]
    base = float(base_vals[k]) if len(base_vals) > k else 0.0
    etiquetas = [f"{ORIG_VARS[j]} = {X_te.iloc[i][ORIG_VARS[j]]}" for j in orden]
    fig = go.Figure(go.Waterfall(orientation="h", y=etiquetas, x=[ph[j] for j in orden], measure=["relative"] * len(orden),
                                 increasing=dict(marker_color="#1e8449"), decreasing=dict(marker_color="#c0392b")))
    real = CLASES[int(y_te[i])]; pred = CLASES[int(P_best[i].argmax())]
    fig.update_layout(title=f"SHAP (waterfall/force) hogar {X_te.index[i]} | clase explicada: {clase} | real: {real} | predicho: {pred} | valor base={base:.2f}", height=520, title_font_size=13)
    return fig
reg_fig("clf_shap_force_ejemplo", fig_shap_force(0, "Sí"))
try:
    ejemplo = shap.force_plot(float(base_vals[2]), SV[0, :, 2], Xt_te[0], feature_names=nombres_ohe, matplotlib=False)
    shap.save_html(f"{DIRS['graficos']}/clf_shap_force_plot_nativo.html", ejemplo)
except Exception as e:
    print("  (force plot nativo de SHAP no exportado:", str(e)[:70], ")")

# ---- Fairness (predicciones fuera de muestra sobre los 1000 hogares: grupos más grandes y estables que el set de prueba)
P_oof = cross_val_predict(clone(MEJOR_PIPE), X_all, y_all, cv=CV5, method="predict_proba", n_jobs=1)
pred_oof = P_oof.argmax(axis=1)
df_f = df[["estrato", "zona", "jefe_hogar_genero"]].copy(); df_f["estrato"] = "E" + df_f["estrato"].astype(str)
df_f["estrato_zona"] = df_f["estrato"] + " | " + df_f["zona"]
df_f["real"] = y_all; df_f["pred"] = pred_oof; df_f["p_si"] = P_oof[:, 2]
def tabla_fairness(col):
    filas = []
    for g, d in df_f.groupby(col):
        yr, yp = d["real"].values, d["pred"].values
        pos_real, pos_pred = (yr == 2), (yp == 2)
        tpr = pos_pred[pos_real].mean() if pos_real.any() else np.nan
        fpr = pos_pred[~pos_real].mean() if (~pos_real).any() else np.nan
        filas.append({"grupo": g, "n": len(d), "tasa_real_Si": pos_real.mean(), "tasa_pred_Si": pos_pred.mean(), "TPR_Si": tpr, "FPR_Si": fpr,
                      "exactitud": (yr == yp).mean(), "f1_macro": f1_score(yr, yp, average="macro")})
    t = pd.DataFrame(filas).set_index("grupo")
    resumen = {"dif_paridad_demografica (max-min tasa_pred_Si)": t["tasa_pred_Si"].max() - t["tasa_pred_Si"].min(),
               "dif_oportunidad_igual (max-min TPR_Si)": t["TPR_Si"].max() - t["TPR_Si"].min(),
               "impacto_dispar (min/max tasa_pred_Si)": t["tasa_pred_Si"].min() / max(t["tasa_pred_Si"].max(), 1e-9),
               "dif_exactitud (max-min)": t["exactitud"].max() - t["exactitud"].min()}
    return t.round(3), {k: round(float(v), 3) for k, v in resumen.items()}
FAIR = {}
for col in ["estrato", "zona", "estrato_zona", "jefe_hogar_genero"]:
    FAIR[col] = tabla_fairness(col)
    reg_tabla(f"fairness_{col}", FAIR[col][0], index=True)
def fig_fairness(col):
    t, r = FAIR[col]
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Tasa real vs predicha de 'Sí'", "TPR / FPR de 'Sí' y exactitud"))
    fig.add_bar(x=t.index, y=t["tasa_real_Si"], name="Real", marker_color="#aab7b8", row=1, col=1)
    fig.add_bar(x=t.index, y=t["tasa_pred_Si"], name="Predicha", marker_color="#2471a3", row=1, col=1)
    fig.add_bar(x=t.index, y=t["TPR_Si"], name="TPR (sensibilidad)", marker_color="#1e8449", row=1, col=2)
    fig.add_bar(x=t.index, y=t["FPR_Si"], name="FPR", marker_color="#c0392b", row=1, col=2)
    fig.add_bar(x=t.index, y=t["exactitud"], name="Exactitud", marker_color="#e6a817", row=1, col=2)
    alerta = "  ⚠ impacto dispar < 0.8" if r["impacto_dispar (min/max tasa_pred_Si)"] < 0.8 else ""
    fig.update_layout(barmode="group", height=470, title=f"Equidad por {col} | Δ paridad={r['dif_paridad_demografica (max-min tasa_pred_Si)']}, "
                                                        f"Δ oportunidad={r['dif_oportunidad_igual (max-min TPR_Si)']}{alerta}", title_font_size=13)
    return fig
reg_fig("fairness_estrato", fig_fairness("estrato"), clave=True); reg_fig("fairness_zona", fig_fairness("zona"))
print("Fairness por estrato:", FAIR["estrato"][1]); print("Fairness por zona:", FAIR["zona"][1])

# ---- entrenamiento final sobre el 100% de los datos para el simulador (el modelo evaluado queda en MODELO_FINAL_CLF)
log("Clasificación completada")

# =============================================================================
# BLOQUE 4. REGRESIÓN: IMPACTO ECONÓMICO
# =============================================================================
log("BLOQUE 4: Regresión (ahorro anual y retorno simple)")
# Variables que un hogar/técnico puede aportar ANTES de saber el ahorro (sin fugas: sin ahorro, inversión, VPN, TIR, retorno, generación).
NUM_REG = ["estrato", "num_personas", "consumo_kwh_mes", "tarifa_kwh_cop", "area_cubierta_m2", "inclinacion_cubierta_grados",
           "num_paneles_estimados", "potencia_panel_w", "ratio_diurno_6_17"]
CAT_REG = ["zona", "tipo_cubierta", "orientacion_cubierta", "sombras_presentes"]
FEATS_REG = NUM_REG + CAT_REG
TARGETS_REG = {"ahorro_anual_cop": "Ahorro anual (COP)", "retorno_simple_anios": "Retorno simple (años)"}
Xr = df[FEATS_REG].copy()
KF5 = KFold(5, shuffle=True, random_state=SEED)
REG = {}
for tgt in TARGETS_REG:
    yr = df[tgt].values
    Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=SEED, stratify=df["estrato"])
    modelos = {
        "Ridge (línea base)": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(n_estimators=500, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
        "XGBoost": XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.04, subsample=0.85, colsample_bytree=0.85, random_state=SEED, n_jobs=-1, verbosity=0, tree_method="hist"),
        "MLP (64-32)": TransformedTargetRegressor(MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-2, early_stopping=True, max_iter=2000, random_state=SEED), transformer=StandardScaler())}
    filas, fitted, preds = [], {}, {}
    for nombre, mdl in modelos.items():
        pipe = Pipeline([("prep", nuevo_preproceso(NUM_REG, CAT_REG)), ("reg", mdl)])
        cv = cross_validate(pipe, Xr_tr, yr_tr, cv=KF5, scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"}, n_jobs=1)
        pipe.fit(Xr_tr, yr_tr); p = pipe.predict(Xr_te)
        filas.append({"modelo": nombre, "cv_R2": cv["test_r2"].mean(), "cv_MAE": -cv["test_mae"].mean(), "cv_RMSE": -cv["test_rmse"].mean(),
                      "test_R2": r2_score(yr_te, p), "test_MAE": mean_absolute_error(yr_te, p), "test_RMSE": float(np.sqrt(mean_squared_error(yr_te, p)))})
        fitted[nombre] = pipe; preds[nombre] = p
    tab = pd.DataFrame(filas).set_index("modelo").round(4).sort_values("cv_R2", ascending=False)
    reg_tabla(f"regresion_{tgt}", tab, index=True)
    mejor = tab.index[0]
    # reentrenar el mejor con el 100% de los datos para el simulador
    final = clone(fitted[mejor]).fit(Xr, yr)
    REG[tgt] = dict(tabla=tab, mejor=mejor, modelo=final, y_te=yr_te, pred=preds[mejor], X_te=Xr_te, fitted=fitted)
    log(f"  {TARGETS_REG[tgt]}: mejor={mejor} | CV R²={tab.loc[mejor, 'cv_R2']:.3f} | test R²={tab.loc[mejor, 'test_R2']:.3f} | MAE={tab.loc[mejor, 'test_MAE']:.2f}")
    # gráficos
    pr = preds[mejor]; res = yr_te - pr
    fig = make_subplots(rows=2, cols=2, subplot_titles=("Real vs predicho", "Residuos vs predicho", "Histograma de residuos", "QQ-plot de residuos"))
    lim = [min(yr_te.min(), pr.min()), max(yr_te.max(), pr.max())]
    fig.add_trace(go.Scatter(x=yr_te, y=pr, mode="markers", marker=dict(color=Xr_te["estrato"].map(COLOR_E), size=6, opacity=0.7), showlegend=False,
                             text=[f"E{e}" for e in Xr_te["estrato"]]), row=1, col=1)
    fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=pr, y=res, mode="markers", marker=dict(size=6, opacity=0.7, color="#2471a3"), showlegend=False), row=1, col=2)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)
    fig.add_trace(go.Histogram(x=res, nbinsx=30, marker_color="#8e44ad", showlegend=False), row=2, col=1)
    (osm, osr), (slope, intercept, r_) = stats.probplot(res, dist="norm")
    fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", marker=dict(size=5, color="#1e8449"), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=osm, y=slope * np.asarray(osm) + intercept, mode="lines", line=dict(color="gray", dash="dash"), showlegend=False), row=2, col=2)
    sw_p = stats.shapiro(res)[1]
    fig.update_layout(height=760, title=f"{TARGETS_REG[tgt]} - {mejor} | R²={r2_score(yr_te, pr):.3f} | MAE={mean_absolute_error(yr_te, pr):.2f} | Shapiro-Wilk p={sw_p:.3g}", title_font_size=13)
    reg_fig(f"reg_diagnostico_{tgt}", fig, clave=True)
    imp = permutation_importance(fitted[mejor], Xr_te, yr_te, scoring="r2", n_repeats=10, random_state=SEED, n_jobs=1)
    REG[tgt]["importancia"] = pd.DataFrame({"variable": FEATS_REG, "importancia": imp.importances_mean}).sort_values("importancia", ascending=False)
    reg_tabla(f"regresion_importancia_{tgt}", REG[tgt]["importancia"].round(4))
fig_cmp_reg = go.Figure()
for tgt in TARGETS_REG:
    t = REG[tgt]["tabla"]
    fig_cmp_reg.add_bar(x=t.index, y=t["test_R2"], name=TARGETS_REG[tgt])
fig_cmp_reg.update_layout(barmode="group", title="R² en el set de prueba por modelo y objetivo", height=420, yaxis_range=[0, 1.05])
reg_fig("reg_comparacion", fig_cmp_reg)
COSTO_KWP_REF = float(df["costo_kwp_cop"].median())
log("Regresión completada")

# =============================================================================
# BLOQUE 5. APRENDIZAJE NO SUPERVISADO: SEGMENTACIÓN DE HOGARES
# =============================================================================
log("BLOQUE 5: Clustering")
CL_FEATS = ["consumo_kwh_mes", "ingreso_mensual_cop", "num_personas", "area_cubierta_m2", "generacion_potencial_kwp", "porcentaje_ahorro",
            "retorno_simple_anios", "ratio_diurno_6_17", "ratio_noche_18_22", "indice_eficiencia_vivienda", "carga_familiar", "estrato"]
sc_cl = StandardScaler().fit(df[CL_FEATS]); Xc = sc_cl.transform(df[CL_FEATS])
pca_cl = PCA(n_components=2, random_state=SEED).fit(Xc); Zc = pca_cl.transform(Xc)

filas_k = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xc)
    filas_k.append({"k": k, "inercia": km.inertia_, "silhouette": silhouette_score(Xc, km.labels_),
                    "calinski_harabasz": calinski_harabasz_score(Xc, km.labels_), "davies_bouldin": davies_bouldin_score(Xc, km.labels_)})
tab_k = pd.DataFrame(filas_k).round(4); reg_tabla("kmeans_seleccion_k", tab_k)
K_OPT = int(tab_k[(tab_k.k >= 3) & (tab_k.k <= 7)].sort_values("silhouette", ascending=False).iloc[0]["k"])   # k>=3 para segmentos útiles
print(f"K óptimo (máx. silhouette entre 3 y 7): {K_OPT}")
fig_k = make_subplots(rows=1, cols=3, subplot_titles=("Método del codo (inercia)", "Silhouette (↑)", "Davies-Bouldin (↓)"))
fig_k.add_trace(go.Scatter(x=tab_k.k, y=tab_k.inercia, mode="lines+markers", showlegend=False), row=1, col=1)
fig_k.add_trace(go.Scatter(x=tab_k.k, y=tab_k.silhouette, mode="lines+markers", marker_color="#1e8449", showlegend=False), row=1, col=2)
fig_k.add_trace(go.Scatter(x=tab_k.k, y=tab_k.davies_bouldin, mode="lines+markers", marker_color="#c0392b", showlegend=False), row=1, col=3)
for c in (1, 2, 3): fig_k.add_vline(x=K_OPT, line_dash="dash", line_color="gray", row=1, col=c)
fig_k.update_layout(title=f"Selección de k para K-Means (elegido k={K_OPT})", height=420)
reg_fig("kmeans_seleccion_k", fig_k, clave=True)

KM = KMeans(n_clusters=K_OPT, n_init=30, random_state=SEED).fit(Xc)
df["cluster_kmeans"] = KM.labels_
medias = df.groupby("cluster_kmeans")[["consumo_kwh_mes", "estrato", "ingreso_mensual_cop"]].mean()
orden_cons = medias["consumo_kwh_mes"].rank(method="first").astype(int)
def _nombre_cluster(c):
    r = int(orden_cons[c]); n = len(medias)
    nivel = "consumo bajo" if r <= n / 3 else ("consumo medio" if r <= 2 * n / 3 else "consumo alto")
    return f"C{c}: {nivel}, estrato≈{medias.loc[c, 'estrato']:.1f}"
NOMBRES_CL = {c: _nombre_cluster(c) for c in medias.index}
df["segmento"] = df["cluster_kmeans"].map(NOMBRES_CL)
perfil = df.groupby("segmento").agg(hogares=("id_hogar", "size"), consumo_kwh_mes=("consumo_kwh_mes", "mean"), ingreso_cop=("ingreso_mensual_cop", "mean"),
                                    personas=("num_personas", "mean"), area_cubierta_m2=("area_cubierta_m2", "mean"), kwp=("generacion_potencial_kwp", "mean"),
                                    pct_ahorro=("porcentaje_ahorro", "mean"), retorno_anios=("retorno_simple_anios", "mean"),
                                    ratio_diurno=("ratio_diurno_6_17", "mean"), pct_rural=("zona", lambda s: 100 * (s == "Rural").mean()),
                                    pct_dispuestos_Si=("dispuesto_instalar_paneles", lambda s: 100 * (s == "Sí").mean()),
                                    vpn_positivo_pct=("vpn_10_anios_cop", lambda s: 100 * (s > 0).mean())).round(2)
reg_tabla("kmeans_perfil_segmentos", perfil, index=True)
print(perfil.to_string())
fig_clu = px.scatter(x=Zc[:, 0], y=Zc[:, 1], color=df["segmento"], opacity=0.75, labels={"x": f"PC1 ({pca_cl.explained_variance_ratio_[0]*100:.1f}%)", "y": f"PC2 ({pca_cl.explained_variance_ratio_[1]*100:.1f}%)"},
                     title=f"K-Means (k={K_OPT}) proyectado en PCA 2D | silhouette={silhouette_score(Xc, KM.labels_):.3f}")
cen2 = pca_cl.transform(KM.cluster_centers_)
fig_clu.add_trace(go.Scatter(x=cen2[:, 0], y=cen2[:, 1], mode="markers+text", text=[f"C{i}" for i in range(K_OPT)], marker=dict(symbol="x", size=14, color="black"), name="Centroides", textposition="top center"))
fig_clu.update_layout(height=560)
reg_fig("kmeans_pca", fig_clu, clave=True)
zc = pd.DataFrame(StandardScaler().fit_transform(df[CL_FEATS]), columns=CL_FEATS).groupby(df["segmento"].values).mean().round(2)
fig_heat = px.imshow(zc.T, color_continuous_scale="RdBu_r", zmin=-2, zmax=2, text_auto=True, aspect="auto", title="Perfil estandarizado de cada segmento (z-score de la media)")
fig_heat.update_layout(height=520)
reg_fig("kmeans_perfil_heatmap", fig_heat)
horas_seg = df.groupby("segmento")[HORAS].mean().T * 1000
fig_hseg = px.line(horas_seg, labels={"index": "Hora", "value": "Wh por hora", "segmento": "Segmento"}, title="Perfil horario medio por segmento")
fig_hseg.update_layout(height=450); reg_fig("kmeans_perfil_horario", fig_hseg)

# ---- Jerárquico (Ward) sobre todos los hogares + dendrograma truncado sobre muestra
AGG = AgglomerativeClustering(n_clusters=K_OPT, linkage="ward").fit(Xc)
df["cluster_jerarquico"] = AGG.labels_
ARI = adjusted_rand_score(df["cluster_kmeans"], df["cluster_jerarquico"])
sil_h = silhouette_score(Xc, AGG.labels_)
print(f"Jerárquico Ward k={K_OPT}: silhouette={sil_h:.3f} | ARI vs K-Means={ARI:.3f}")
idx_h = np.sort(np.random.default_rng(SEED).choice(len(df), 120, replace=False))
fig_dend = ff.create_dendrogram(Xc[idx_h], labels=[f"E{df.estrato.iloc[i]}-{df.id_hogar.iloc[i][1:]}" for i in idx_h], linkagefun=lambda x: linkage(x, "ward"))
fig_dend.update_layout(title=f"Dendrograma Ward (120 hogares) | Jerárquico k={K_OPT}: silhouette={sil_h:.3f}, ARI vs K-Means={ARI:.2f}", height=620, xaxis_tickangle=90)
reg_fig("dendrograma_clustering", fig_dend, clave=True)

# ---- DBSCAN: hogares atípicos
pca_db = PCA(n_components=0.90, random_state=SEED).fit(Xc); Zdb = pca_db.transform(Xc)
MIN_S = 8
kd = np.sort(NearestNeighbors(n_neighbors=MIN_S).fit(Zdb).kneighbors(Zdb)[0][:, -1])
mejor_db = None
for q in (0.90, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99):
    eps = float(np.quantile(kd, q)); lab = DBSCAN(eps=eps, min_samples=MIN_S).fit_predict(Zdb)
    frac = float((lab == -1).mean())
    if 0.015 <= frac <= 0.10:
        mejor_db = (q, eps, lab, frac); break
if mejor_db is None:
    eps = float(np.quantile(kd, 0.95)); lab = DBSCAN(eps=eps, min_samples=MIN_S).fit_predict(Zdb); mejor_db = (0.95, eps, lab, float((lab == -1).mean()))
q_db, EPS_DB, lab_db, frac_db = mejor_db
df["dbscan"] = lab_db; df["atipico_dbscan"] = (lab_db == -1)
print(f"DBSCAN: eps={EPS_DB:.3f} (percentil {q_db:.2f} de la distancia al {MIN_S}º vecino) | clusters={len(set(lab_db)) - (1 if -1 in lab_db else 0)} | atípicos={int((lab_db == -1).sum())} ({frac_db*100:.1f}%)")
fig_kd = go.Figure(go.Scatter(y=kd, mode="lines")); fig_kd.add_hline(y=EPS_DB, line_dash="dash", line_color="red", annotation_text=f"eps={EPS_DB:.2f}")
fig_kd.update_layout(title=f"Distancia al {MIN_S}º vecino más cercano (ordenada) - criterio para eps", height=380); reg_fig("dbscan_kdistancia", fig_kd)
dbp = pd.DataFrame({"PC1": Zc[:, 0], "PC2": Zc[:, 1], "tipo": np.where(lab_db == -1, "Atípico", "Normal"), "estrato": df["estrato"].astype(str), "id": df["id_hogar"]})
fig_db = px.scatter(dbp, x="PC1", y="PC2", color="tipo", symbol="estrato", color_discrete_map={"Atípico": "#c0392b", "Normal": "#aab7b8"}, hover_name="id", opacity=0.8,
                    title=f"DBSCAN: {int((lab_db==-1).sum())} hogares atípicos ({frac_db*100:.1f}%)")
fig_db.update_layout(height=520); reg_fig("dbscan_atipicos", fig_db, clave=True)
atip = df.loc[df.atipico_dbscan, ["id_hogar", "estrato", "zona", "num_personas", "consumo_kwh_mes", "ingreso_mensual_cop", "generacion_potencial_kwp",
                                  "porcentaje_ahorro", "retorno_simple_anios", "dispuesto_instalar_paneles"]].round(2)
reg_tabla("dbscan_hogares_atipicos", atip)
reg_tabla("hogares_con_segmentos", df[["id_hogar", "estrato", "zona", "cluster_kmeans", "segmento", "cluster_jerarquico", "dbscan", "atipico_dbscan"]])
log("Clustering completado")

# =============================================================================
# BLOQUE 6. SERIES TEMPORALES Y PRONÓSTICO
# =============================================================================
log("BLOQUE 6: Series temporales")
FECHA_INI, FECHA_FIN = "2024-11-01", "2026-10-31"       # 24 meses completos
FECHA_CORTE_BT = "2026-04-30"                            # backtest: entrenar 18 meses, validar 6
H_FINAL = 365                                            # pronóstico final: 12 meses (nov-2026 a oct-2027)

def generar_series_sinteticas(df_, semilla=SEED):
    """Serie diaria sintética de 2 años (consumo y generación FV por hogar, kWh/día), calibrada con el dataset:
    nivel base = consumo y kWp medios de la muestra; estacionalidad bimodal de lluvias de Popayán (picos de lluvia en
    abril-mayo y octubre-noviembre), pico navideño de diciembre, efecto fin de semana, tendencia suave, ruido AR(1),
    nubosidad diaria y anomalías inyectadas (con etiqueta) para validar la detección."""
    rg = np.random.default_rng(semilla)
    fechas = pd.date_range(FECHA_INI, FECHA_FIN, freq="D"); n = len(fechas); doy = fechas.dayofyear.values
    cons0 = float(df_["consumo_kwh_dia_promedio"].mean()); kwp0 = float(df_["generacion_potencial_kwp"].mean())
    hsp0 = float(df_["horas_sol_pico"].mean()); amp_est = float(np.clip(df_["variacion_estacional_pct"].abs().mean() / 100, 0.02, 0.12))
    lluvia = np.cos(4 * np.pi * (doy - 120) / 365.0)                       # +1 en ~30-abr y ~30-oct (temporadas lluviosas)
    navidad = np.exp(-0.5 * ((((doy - 355) + 182) % 365 - 182) / 10.0) ** 2)
    findes = np.where(fechas.dayofweek >= 5, 1.06, 1.0)
    tend = 1 + 0.03 * np.arange(n) / 365.0
    ar = np.zeros(n)
    for t in range(1, n): ar[t] = 0.55 * ar[t - 1] + rg.normal(0, 0.028)
    consumo = cons0 * (1 + amp_est * lluvia + 0.07 * navidad) * findes * tend * np.exp(ar)
    etiqueta = np.zeros(n, dtype=int)
    idx_an = rg.choice(np.arange(20, n - 20), 12, replace=False)
    for j, t in enumerate(idx_an):
        consumo[t] *= rg.uniform(1.45, 1.9) if j % 4 != 3 else rg.uniform(0.45, 0.6); etiqueta[t] = 1
    nubes = np.clip(1 - 0.10 * (lluvia > 0.2) * lluvia + rg.normal(0, 0.12, n), 0.30, 1.20)
    gen = kwp0 * hsp0 * 0.78 * (1 - 0.10 * np.clip(lluvia, 0, 1)) * np.clip(nubes, 0.3, 1.2) * (1 - 0.005 * np.arange(n) / 365.0)
    return pd.DataFrame({"consumo_kwh_dia": consumo, "generacion_kwh_dia": gen, "anomalia_real": etiqueta}, index=fechas)

serie_d = generar_series_sinteticas(df)
serie_d["balance_kwh_dia"] = serie_d["generacion_kwh_dia"] - serie_d["consumo_kwh_dia"]
serie_m = serie_d[["consumo_kwh_dia", "generacion_kwh_dia"]].resample("MS").sum().rename(columns={"consumo_kwh_dia": "consumo_kwh_mes", "generacion_kwh_dia": "generacion_kwh_mes"})
serie_m["autosuficiencia_pct"] = 100 * serie_m["generacion_kwh_mes"] / serie_m["consumo_kwh_mes"]
reg_tabla("serie_diaria_sintetica", serie_d.round(4), index=True); reg_tabla("serie_mensual_sintetica", serie_m.round(3), index=True)
print(f"Serie sintética: {len(serie_d)} días ({serie_d.index.min().date()} a {serie_d.index.max().date()}), {len(serie_m)} meses, "
      f"{int(serie_d['anomalia_real'].sum())} anomalías inyectadas")

SERIES = {"Consumo": ("consumo_kwh_dia", "consumo_kwh_mes"), "Generación FV": ("generacion_kwh_dia", "generacion_kwh_mes")}
fig_ts = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Serie diaria (kWh/día por hogar)", "Serie mensual (kWh/mes por hogar)"))
fig_ts.add_trace(go.Scatter(x=serie_d.index, y=serie_d["consumo_kwh_dia"], name="Consumo", line=dict(color="#c0392b", width=1)), row=1, col=1)
fig_ts.add_trace(go.Scatter(x=serie_d.index, y=serie_d["generacion_kwh_dia"], name="Generación FV", line=dict(color="#e6a817", width=1)), row=1, col=1)
fig_ts.add_trace(go.Scatter(x=serie_m.index, y=serie_m["consumo_kwh_mes"], name="Consumo (mes)", mode="lines+markers", line=dict(color="#c0392b")), row=2, col=1)
fig_ts.add_trace(go.Scatter(x=serie_m.index, y=serie_m["generacion_kwh_mes"], name="Generación (mes)", mode="lines+markers", line=dict(color="#e6a817")), row=2, col=1)
fig_ts.update_layout(height=620, title="Series sintéticas calibradas con el dataset: consumo y generación por hogar promedio")
reg_fig("ts_series", fig_ts, clave=True)

# ---- STL
def fig_stl(serie="Consumo", frecuencia="Diaria"):
    cd, cm_ = SERIES[serie]
    if frecuencia == "Diaria":
        y = serie_d[cd]; per = 7; res = STL(y, period=per, robust=True).fit(); tit = "STL diaria (periodo = 7 días)"
    else:
        y = serie_m[cm_]; per = 12; res = STL(y, period=per, robust=True).fit(); tit = "STL mensual (periodo = 12 meses)"
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=("Observada", "Tendencia", "Estacionalidad", "Residuo"), vertical_spacing=0.06)
    for i, (comp, col) in enumerate(((y, "#2471a3"), (res.trend, "#1e8449"), (res.seasonal, "#8e44ad"), (res.resid, "#c0392b")), start=1):
        fig.add_trace(go.Scatter(x=y.index, y=comp, mode="lines", line=dict(color=col, width=1.3), showlegend=False), row=i, col=1)
    fe = max(0.0, 1 - res.resid.var() / (res.resid + res.seasonal).var()); ft = max(0.0, 1 - res.resid.var() / (res.resid + res.trend).var())
    fig.update_layout(height=760, title=f"{serie} - {tit} | fuerza estacional={fe:.2f}, fuerza de tendencia={ft:.2f}")
    return fig
reg_fig("ts_stl_consumo_diaria", fig_stl("Consumo", "Diaria"), clave=True); reg_fig("ts_stl_consumo_mensual", fig_stl("Consumo", "Mensual"))

# ---- ACF / PACF
def fig_acf_pacf(serie="Consumo", frecuencia="Diaria"):
    cd, cm_ = SERIES[serie]
    if frecuencia == "Diaria":
        y = serie_d[cd].diff().dropna(); nl = 40; tit = "diaria, diferenciada d=1"
    else:
        y = serie_m[cm_]; nl = 10; tit = "mensual (24 puntos)"
    a = acf(y, nlags=nl, fft=True); p = pacf(y, nlags=nl, method="ywm"); ci = 1.96 / np.sqrt(len(y))
    fig = make_subplots(rows=1, cols=2, subplot_titles=("ACF", "PACF"))
    for c, v in ((1, a), (2, p)):
        fig.add_trace(go.Bar(x=list(range(len(v))), y=v, marker_color="#2471a3", showlegend=False), row=1, col=c)
        fig.add_hline(y=ci, line_dash="dash", line_color="red", row=1, col=c); fig.add_hline(y=-ci, line_dash="dash", line_color="red", row=1, col=c)
        fig.update_xaxes(title_text="Rezago", row=1, col=c)
    fig.update_layout(height=430, title=f"{serie}: ACF y PACF ({tit}); bandas ±1.96/√n")
    return fig
reg_fig("ts_acf_pacf_consumo", fig_acf_pacf("Consumo", "Diaria"), clave=True)

# ---- Modelos de pronóstico (SARIMAX con exógenas de Fourier  vs  Prophet), evaluados con backtest de 6 meses
def fourier(index, K=3):
    t = np.arange(len(index)); doy = index.dayofyear.values
    cols = {}
    for k in range(1, K + 1):
        cols[f"s{k}"] = np.sin(2 * np.pi * k * doy / 365.25); cols[f"c{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    return pd.DataFrame(cols, index=index)

def ajustar_sarimax(y):
    Xf = fourier(y.index)
    for orden, sord in (((1, 0, 1), (1, 0, 1, 7)), ((1, 0, 0), (1, 0, 0, 7)), ((1, 0, 0), (0, 0, 0, 0))):
        try:
            m = SARIMAX(y, exog=Xf, order=orden, seasonal_order=sord, trend="c", enforce_stationarity=False, enforce_invertibility=False)
            return m.fit(disp=False, maxiter=200), orden, sord
        except Exception:
            continue
    raise RuntimeError("SARIMAX no convergió")

def pronostico_sarimax(res, y_index_fin, h, n_sim=300):
    fidx = pd.date_range(y_index_fin + pd.Timedelta(days=1), periods=h, freq="D"); Xf = fourier(fidx)
    mean = res.get_forecast(steps=h, exog=Xf).predicted_mean.values
    try:
        sim = np.asarray(res.simulate(nsimulations=h, repetitions=n_sim, anchor="end", exog=Xf)).reshape(h, -1)
    except Exception:
        sd = float(np.std(res.resid[10:])); sim = mean[:, None] + np.random.default_rng(SEED).normal(0, sd, (h, n_sim))
    return fidx, mean, sim

def ajustar_prophet(y):
    if not PROPHET_OK: return None
    d = pd.DataFrame({"ds": y.index, "y": y.values})
    m = Prophet(yearly_seasonality=4, weekly_seasonality=True, daily_seasonality=False, seasonality_mode="multiplicative", interval_width=0.90,
                changepoint_prior_scale=0.05, uncertainty_samples=300)
    m.fit(d); return m

def pronostico_prophet(m, h):
    fut = m.make_future_dataframe(periods=h, freq="D"); fc = m.predict(fut).iloc[-h:]
    try:
        samp = np.asarray(m.predictive_samples(fut)["yhat"])[-h:, :]
    except Exception:
        samp = None
    return pd.DatetimeIndex(fc["ds"]), fc["yhat"].values, samp

def a_mensual(idx, valores):
    return pd.Series(valores, index=idx).resample("MS").sum()

def a_mensual_bandas(idx, sims, lo=5, hi=95):
    if sims is None: return None, None
    df_s = pd.DataFrame(sims, index=idx).resample("MS").sum()
    return np.percentile(df_s.values, lo, axis=1), np.percentile(df_s.values, hi, axis=1)

def evaluar(real_m, pred_m):
    ok = real_m.index.intersection(pred_m.index)
    r, p = real_m.loc[ok].values, pred_m.loc[ok].values
    return {"MAE_mensual": mean_absolute_error(r, p), "RMSE_mensual": float(np.sqrt(mean_squared_error(r, p))), "MAPE_%": float(np.mean(np.abs((r - p) / r)) * 100)}

TS = {}
filas_bt = []
for nombre, (cd, cm_) in SERIES.items():
    ys = serie_d[cd]; ts_train = ys[:FECHA_CORTE_BT]; ts_test = ys[FECHA_CORTE_BT:].iloc[1:]
    real_m = ts_test.resample("MS").sum()
    # línea base estacional: mismo mes del año anterior (sumas mensuales)
    ym = serie_d[cd].resample("MS").sum(); base_m = pd.Series([ym.get(i - pd.DateOffset(years=1), np.nan) for i in real_m.index], index=real_m.index)
    filas_bt.append({"serie": nombre, "modelo": "Naive estacional (mismo mes año previo)", **evaluar(real_m, base_m.dropna())})
    t1 = time.time(); res_bt, o_, so_ = ajustar_sarimax(ts_train)
    fidx, mean, sim = pronostico_sarimax(res_bt, ts_train.index[-1], len(ts_test), n_sim=100)
    filas_bt.append({"serie": nombre, "modelo": f"SARIMAX{o_}x{so_}+Fourier", **evaluar(real_m, a_mensual(fidx, mean))})
    if PROPHET_OK:
        mp = ajustar_prophet(ts_train); pidx, pm, _ = pronostico_prophet(mp, len(ts_test))
        filas_bt.append({"serie": nombre, "modelo": "Prophet", **evaluar(real_m, a_mensual(pidx, pm))})
    # ajuste final con toda la serie y pronóstico a 12 meses
    res_f, o_, so_ = ajustar_sarimax(ys)
    fidx, mean, sim = pronostico_sarimax(res_f, ys.index[-1], H_FINAL, n_sim=300)
    m_sar = a_mensual(fidx, mean); lo_s, hi_s = a_mensual_bandas(fidx, sim)
    pro = None
    if PROPHET_OK:
        mp_f = ajustar_prophet(ys); pidx, pm, psamp = pronostico_prophet(mp_f, H_FINAL)
        m_pro = a_mensual(pidx, pm); lo_p, hi_p = a_mensual_bandas(pidx, psamp)
        pro = dict(modelo=mp_f, serie_mensual=m_pro, lo=lo_p, hi=hi_p, diario=pd.Series(pm, index=pidx))
    TS[nombre] = dict(sarimax=res_f, orden=(o_, so_), sar_mensual=m_sar, sar_lo=lo_s, sar_hi=hi_s, prophet=pro, hist_m=ym, sar_diario=pd.Series(mean, index=fidx))
    log(f"  {nombre}: SARIMAX{o_}x{so_} y Prophet ajustados ({time.time()-t1:.0f}s)")
tab_bt = pd.DataFrame(filas_bt).round(3); reg_tabla("ts_backtest_6_meses", tab_bt)
print(tab_bt.to_string(index=False))

def fig_pronostico(serie="Consumo"):
    d = TS[serie]; fig = go.Figure()
    fig.add_trace(go.Scatter(x=d["hist_m"].index, y=d["hist_m"].values, mode="lines+markers", name="Histórico (mensual)", line=dict(color="black")))
    if d["sar_lo"] is not None:
        fig.add_trace(go.Scatter(x=list(d["sar_mensual"].index) + list(d["sar_mensual"].index[::-1]), y=list(d["sar_hi"]) + list(d["sar_lo"][::-1]),
                                 fill="toself", fillcolor="rgba(36,113,163,0.15)", line=dict(width=0), name="SARIMAX IC 90%", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=d["sar_mensual"].index, y=d["sar_mensual"].values, mode="lines+markers", name="SARIMAX+Fourier", line=dict(color="#2471a3")))
    if d["prophet"] is not None:
        p = d["prophet"]
        if p["lo"] is not None:
            fig.add_trace(go.Scatter(x=list(p["serie_mensual"].index) + list(p["serie_mensual"].index[::-1]), y=list(p["hi"]) + list(p["lo"][::-1]),
                                     fill="toself", fillcolor="rgba(192,57,43,0.15)", line=dict(width=0), name="Prophet IC 90%", hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=p["serie_mensual"].index, y=p["serie_mensual"].values, mode="lines+markers", name="Prophet", line=dict(color="#c0392b", dash="dash")))
    fig.update_layout(title=f"Pronóstico a 12 meses (nov-2026 a oct-2027): {serie} por hogar (kWh/mes)", height=520, yaxis_title="kWh/mes", hovermode="x unified")
    return fig
reg_fig("ts_pronostico_consumo", fig_pronostico("Consumo"), clave=True); reg_fig("ts_pronostico_generacion", fig_pronostico("Generación FV"))
filas_pron = []
for nm, d in TS.items():
    base = pd.DataFrame({"mes": d["sar_mensual"].index.strftime("%Y-%m"), "serie": nm, "SARIMAX": d["sar_mensual"].values.round(2)})
    if d["prophet"] is not None: base["Prophet"] = d["prophet"]["serie_mensual"].values.round(2)
    filas_pron.append(base)
reg_tabla("ts_pronostico_12_meses", pd.concat(filas_pron))

# ---- Detección de anomalías con Isolation Forest sobre el consumo diario
ys_an = serie_d["consumo_kwh_dia"]; stl_d = STL(ys_an, period=7, robust=True).fit()
mediana_mov = ys_an.rolling(15, center=True, min_periods=5).median()
feat_an = pd.DataFrame({"residuo_stl": stl_d.resid, "z_robusto": (ys_an - mediana_mov) / (1.4826 * (ys_an - mediana_mov).abs().median()),
                        "desv_rel_15d": (ys_an - mediana_mov) / mediana_mov, "dif_1d": ys_an.diff().fillna(0)}, index=ys_an.index)
IFO = IsolationForest(n_estimators=400, contamination=0.02, random_state=SEED).fit(feat_an)
serie_d["anomalia_detectada"] = (IFO.predict(feat_an) == -1).astype(int)
serie_d["score_anomalia"] = -IFO.score_samples(feat_an)
real = serie_d["anomalia_real"].values; det = serie_d["anomalia_detectada"].values
pr_, rc_, f1_ = precision_score(real, det, zero_division=0), recall_score(real, det, zero_division=0), f1_score(real, det, zero_division=0)
print(f"Isolation Forest: detectadas={int(det.sum())} | precisión={pr_:.2f} recall={rc_:.2f} F1={f1_:.2f} (contra anomalías inyectadas)")
reg_tabla("ts_anomalias_detectadas", serie_d.loc[serie_d.anomalia_detectada == 1, ["consumo_kwh_dia", "anomalia_real", "score_anomalia"]].round(3), index=True)
fig_an = go.Figure()
fig_an.add_trace(go.Scatter(x=ys_an.index, y=ys_an.values, mode="lines", name="Consumo diario", line=dict(color="#7f8c8d", width=1)))
tp = serie_d[(serie_d.anomalia_detectada == 1) & (serie_d.anomalia_real == 1)]; fp = serie_d[(serie_d.anomalia_detectada == 1) & (serie_d.anomalia_real == 0)]
fn = serie_d[(serie_d.anomalia_detectada == 0) & (serie_d.anomalia_real == 1)]
fig_an.add_trace(go.Scatter(x=tp.index, y=tp.consumo_kwh_dia, mode="markers", name="Detectada y real", marker=dict(color="#1e8449", size=10, symbol="diamond")))
fig_an.add_trace(go.Scatter(x=fp.index, y=fp.consumo_kwh_dia, mode="markers", name="Falsa alarma", marker=dict(color="#e6a817", size=9, symbol="x")))
fig_an.add_trace(go.Scatter(x=fn.index, y=fn.consumo_kwh_dia, mode="markers", name="No detectada", marker=dict(color="#c0392b", size=10, symbol="circle-open", line=dict(width=2))))
fig_an.update_layout(title=f"Isolation Forest (contaminación 2%) | precisión={pr_:.2f}, recall={rc_:.2f}, F1={f1_:.2f}", height=500, yaxis_title="kWh/día")
reg_fig("ts_anomalias", fig_an, clave=True)
log("Series temporales completadas")

# =============================================================================
# BLOQUE 7. ANALÍTICA PRESCRIPTIVA: ASIGNACIÓN ÓPTIMA DE SUBSIDIOS (PuLP)
# =============================================================================
log("BLOQUE 7: Optimización de presupuesto")
POB_MUNICIPIO, PERSONAS_HOGAR = 320_000, 3.6
HOGARES_MUN = int(POB_MUNICIPIO / PERSONAS_HOGAR)
PESO_ESTRATO = {1: 0.65, 2: 0.20, 3: 0.15}                    # distribución del proyecto restringida a estratos 1-3
FE_CO2_KG_KWH = 0.126                                          # factor de emisión del SIN (kgCO2/kWh): valor de referencia, VERIFICAR con UPME
VIDA_UTIL, DEGRAD = 25, 0.005
FACTOR_VIDA = sum((1 - DEGRAD) ** t for t in range(VIDA_UTIL))
SUBSIDIO_DEF = {1: 0.80, 2: 0.50, 3: 0.20}                    # % de la inversión que cubre el municipio (parámetro de política, editable)

PAR = {}
for s in (1, 2, 3):
    d = df[df.estrato == s]; dsi = d[d.dispuesto_instalar_paneles == "Sí"]
    base = dsi if len(dsi) >= 10 else d
    p_si = float((d.dispuesto_instalar_paneles == "Sí").mean())
    PAR[s] = dict(inversion=float(base["inversion_estimada_cop"].mean()), ahorro_anual=float(base["ahorro_anual_cop"].mean()),
                  gen_anual_kwh=float(base["generacion_mensual_estimada_kwh"].mean() * 12), p_si=p_si,
                  elegibles=int(HOGARES_MUN * PESO_ESTRATO[s] * p_si))
    PAR[s]["energia_vida_kwh"] = PAR[s]["gen_anual_kwh"] * FACTOR_VIDA
    PAR[s]["co2_vida_t"] = PAR[s]["energia_vida_kwh"] * FE_CO2_KG_KWH / 1000.0
    PAR[s]["ahorro_10a_cop"] = PAR[s]["ahorro_anual"] * sum((1.05 * (1 - DEGRAD)) ** t for t in range(10))
reg_tabla("optimizacion_parametros_por_estrato", pd.DataFrame(PAR).T.round(2), index=True)

SOLVER_USADO = {"nombre": None}

def _resolver_pulp(beneficio, costo, caps, presupuesto, min_share):
    """PuLP (<4). Las variables se crean con argumentos POSICIONALES (nombre, cota inf., cota sup., categoría),
    que es lo que evita el TypeError reportado con keywords en algunas versiones."""
    prob = pulp.LpProblem("Subsidios_FV_Popayan", pulp.LpMaximize)
    x = {s: pulp.LpVariable(f"E{s}", 0, caps[s], pulp.LpInteger) for s in (1, 2, 3)}
    prob += pulp.lpSum(beneficio[s] * x[s] for s in x), "Beneficio_total"
    prob += pulp.lpSum(costo[s] * x[s] for s in x) <= presupuesto, "Presupuesto"
    if min_share > 0:
        prob += costo[1] * x[1] - min_share * pulp.lpSum(costo[s] * x[s] for s in x) >= 0, "Equidad_E1"
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[prob.status] != "Optimal":
        return None, pulp.LpStatus[prob.status]
    return {s: int(round(x[s].value() or 0)) for s in x}, "Optimal"

def _resolver_scipy(beneficio, costo, caps, presupuesto, min_share):
    """Respaldo con scipy.optimize.milp (HiGHS): mismo modelo, se usa si PuLP no está disponible o es >= 4."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    c = -np.array([beneficio[s] for s in (1, 2, 3)])
    co = np.array([costo[s] for s in (1, 2, 3)])
    A = [co]; ub = [presupuesto]
    if min_share > 0:
        fila = min_share * co.copy(); fila[0] = -(1 - min_share) * co[0]
        A.append(fila); ub.append(0.0)
    r = milp(c, constraints=LinearConstraint(np.array(A), -np.inf, np.array(ub)), integrality=np.ones(3),
             bounds=Bounds(np.zeros(3), np.array([caps[s] for s in (1, 2, 3)], float)))
    if not r.success:
        return None, r.message
    return {s: int(round(r.x[i])) for i, s in enumerate((1, 2, 3))}, "Optimal"

def _resolver(beneficio, costo, caps, presupuesto, min_share):
    try:
        if int(str(pulp.__version__).split(".")[0]) >= 4:
            raise RuntimeError("PuLP >= 4 usa otra API; se usa scipy.optimize.milp")
        sol = _resolver_pulp(beneficio, costo, caps, presupuesto, min_share); SOLVER_USADO["nombre"] = "PuLP + CBC"
        return sol
    except Exception:
        SOLVER_USADO["nombre"] = "scipy.optimize.milp (HiGHS)"
        return _resolver_scipy(beneficio, costo, caps, presupuesto, min_share)

def optimizar(presupuesto_mm=1000.0, sub1=0.80, sub2=0.50, sub3=0.20, min_share_e1=0.0, w_en=0.5, w_hog=0.3, w_ah=0.2):
    sub = {1: sub1, 2: sub2, 3: sub3}
    tw = max(w_en + w_hog + w_ah, 1e-9); w_en, w_hog, w_ah = w_en / tw, w_hog / tw, w_ah / tw
    costo = {s: PAR[s]["inversion"] * sub[s] for s in sub}
    en_max = max(PAR[s]["energia_vida_kwh"] for s in sub); ah_max = max(PAR[s]["ahorro_10a_cop"] for s in sub)
    beneficio = {s: w_en * PAR[s]["energia_vida_kwh"] / en_max + w_hog * 1.0 + w_ah * PAR[s]["ahorro_10a_cop"] / ah_max for s in sub}
    caps = {s: PAR[s]["elegibles"] for s in sub}
    sol, estado = _resolver(beneficio, costo, caps, presupuesto_mm * 1e6, min_share_e1)
    if sol is None:
        return None, str(estado)
    filas = []
    for s in (1, 2, 3):
        n = int(sol[s])
        filas.append({"estrato": f"E{s}", "subsidio_%": int(sub[s] * 100), "sistemas": n, "subsidio_por_sistema_cop": round(costo[s]), "gasto_municipal_cop": round(costo[s] * n),
                      "inversion_total_cop": round(PAR[s]["inversion"] * n), "hogares_atendidos": n, "energia_vida_util_MWh": round(PAR[s]["energia_vida_kwh"] * n / 1000, 1),
                      "CO2_evitado_t": round(PAR[s]["co2_vida_t"] * n, 1), "ahorro_familias_10a_cop": round(PAR[s]["ahorro_10a_cop"] * n)})
    res = pd.DataFrame(filas)
    tot = {"estrato": "TOTAL", "subsidio_%": np.nan, "sistemas": res.sistemas.sum(), "subsidio_por_sistema_cop": np.nan, "gasto_municipal_cop": res.gasto_municipal_cop.sum(),
           "inversion_total_cop": res.inversion_total_cop.sum(), "hogares_atendidos": res.hogares_atendidos.sum(), "energia_vida_util_MWh": res.energia_vida_util_MWh.sum(),
           "CO2_evitado_t": res.CO2_evitado_t.sum(), "ahorro_familias_10a_cop": res.ahorro_familias_10a_cop.sum()}
    res = pd.concat([res, pd.DataFrame([tot])], ignore_index=True)
    return res, "Optimal"

def fig_optimizacion(res, titulo):
    r = res[res.estrato != "TOTAL"]
    fig = make_subplots(rows=1, cols=3, subplot_titles=("Sistemas subsidiados", "Gasto municipal (millones COP)", "CO₂ evitado en vida útil (t)"))
    cols = [COLOR_E[1], COLOR_E[2], COLOR_E[3]]
    fig.add_bar(x=r.estrato, y=r.sistemas, marker_color=cols, showlegend=False, text=r.sistemas, row=1, col=1)
    fig.add_bar(x=r.estrato, y=(r.gasto_municipal_cop / 1e6).round(1), marker_color=cols, showlegend=False, text=(r.gasto_municipal_cop / 1e6).round(1), row=1, col=2)
    fig.add_bar(x=r.estrato, y=r.CO2_evitado_t, marker_color=cols, showlegend=False, text=r.CO2_evitado_t, row=1, col=3)
    fig.update_layout(height=430, title=titulo, title_font_size=14)
    return fig

def ejecutar_optimizacion(presupuesto_mm, sub1, sub2, sub3, min_e1, w_en, w_hog, w_ah):
    res, est = optimizar(presupuesto_mm, sub1, sub2, sub3, min_e1 / 100.0, w_en, w_hog, w_ah)
    if res is None:
        return f"⚠️ El problema no tiene solución óptima (estado: {est}). Revisa presupuesto, subsidios y restricción de equidad.", go.Figure(), pd.DataFrame()
    t = res.iloc[-1]
    md = (f"### Resultado óptimo (estado: **{est}**)\n"
          f"- Presupuesto: **${presupuesto_mm:,.0f} millones COP** | gasto municipal: **${t.gasto_municipal_cop/1e6:,.1f} M** | sistemas: **{int(t.sistemas):,}**\n"
          f"- Energía evitada en vida útil: **{t.energia_vida_util_MWh:,.0f} MWh** | CO₂ evitado: **{t.CO2_evitado_t:,.0f} tCO₂** (FE={FE_CO2_KG_KWH} kg/kWh, *valor de referencia*)\n"
          f"- Ahorro acumulado de las familias (10 años): **${t.ahorro_familias_10a_cop/1e6:,.0f} M COP**\n"
          f"- Palanca financiera: cada peso de subsidio moviliza **${t.inversion_total_cop/max(t.gasto_municipal_cop,1):.2f}** de inversión total.")
    return md, fig_optimizacion(res, f"Asignación óptima de ${presupuesto_mm:,.0f} M COP | equidad mínima E1 = {min_e1:.0f}%"), res

res_base, _ = optimizar(1000, *SUBSIDIO_DEF.values(), 0.0)
res_eq, _ = optimizar(1000, *SUBSIDIO_DEF.values(), 0.50)
reg_tabla("optimizacion_sin_equidad", res_base); reg_tabla("optimizacion_con_equidad_50", res_eq)
reg_fig("opt_sin_equidad", fig_optimizacion(res_base, "Presupuesto $1.000 M COP: óptimo SIN restricción de equidad"), clave=True)
reg_fig("opt_con_equidad", fig_optimizacion(res_eq, "Presupuesto $1.000 M COP: óptimo con ≥50% del gasto en estrato 1"))
print("Solver utilizado:", SOLVER_USADO["nombre"])
print("Óptimo sin equidad:\n", res_base[["estrato", "sistemas", "gasto_municipal_cop", "CO2_evitado_t"]].to_string(index=False))
print("Óptimo con equidad (>=50% E1):\n", res_eq[["estrato", "sistemas", "gasto_municipal_cop", "CO2_evitado_t"]].to_string(index=False))

# frontera eficiencia-equidad y sensibilidad al presupuesto
filas_f = []
for m_ in range(0, 100, 10):
    r_, _ = optimizar(1000, *SUBSIDIO_DEF.values(), m_ / 100.0)
    if r_ is not None:
        t = r_.iloc[-1]; filas_f.append({"min_share_E1_%": m_, "hogares": t.hogares_atendidos, "CO2_t": t.CO2_evitado_t, "ahorro_familias_M": t.ahorro_familias_10a_cop / 1e6})
tab_f = pd.DataFrame(filas_f); reg_tabla("optimizacion_frontera_equidad", tab_f)
fig_front = make_subplots(specs=[[{"secondary_y": True}]])
fig_front.add_scatter(x=tab_f["min_share_E1_%"], y=tab_f["CO2_t"], mode="lines+markers", name="CO₂ evitado (t)", secondary_y=False)
fig_front.add_scatter(x=tab_f["min_share_E1_%"], y=tab_f["hogares"], mode="lines+markers", name="Hogares atendidos", secondary_y=True)
fig_front.update_xaxes(title_text="Mínimo % del gasto asignado al estrato 1"); fig_front.update_yaxes(title_text="tCO₂", secondary_y=False); fig_front.update_yaxes(title_text="Hogares", secondary_y=True)
fig_front.update_layout(title="Frontera eficiencia-equidad (presupuesto $1.000 M COP)", height=430)
reg_fig("opt_frontera_equidad", fig_front)
filas_p = []
for pm in (250, 500, 750, 1000, 1500, 2000, 3000):
    r_, _ = optimizar(pm, *SUBSIDIO_DEF.values(), 0.30)
    if r_ is not None:
        for _, f in r_[r_.estrato != "TOTAL"].iterrows(): filas_p.append({"presupuesto_M": pm, "estrato": f.estrato, "sistemas": f.sistemas})
fig_pres = px.bar(pd.DataFrame(filas_p), x="presupuesto_M", y="sistemas", color="estrato", color_discrete_map={"E1": COLOR_E[1], "E2": COLOR_E[2], "E3": COLOR_E[3]},
                  title="Sensibilidad al presupuesto (equidad mínima 30% en E1)", barmode="stack")
fig_pres.update_layout(height=430); reg_fig("opt_sensibilidad_presupuesto", fig_pres)
log("Optimización completada")

# =============================================================================
# BLOQUE 8. ECONOMÍA CIRCULAR Y RAEE FOTOVOLTAICO
# =============================================================================
log("BLOQUE 8: Economía circular - RAEE")
COMPOSICION = {"Vidrio": 0.70, "Aluminio": 0.18, "Polímeros (encapsulante/backsheet)": 0.07, "Silicio": 0.03, "Cobre": 0.01, "Otros (Ag, Sn, Pb...)": 0.01}
RECUPERACION = {"Vidrio": 0.85, "Aluminio": 0.95, "Polímeros (encapsulante/backsheet)": 0.0, "Silicio": 0.70, "Cobre": 0.90, "Otros (Ag, Sn, Pb...)": 0.50}
KG_POR_W = 0.052          # masa del panel ≈ 21-29 kg para 400-550 W
MASA_INVERSOR_KG = 8.0
MODOS_RAEE = ["Weibull regular (α=5.38, β=30 años)", "Pérdida temprana ilustrativa (α=2.5, β=18 años)", "Retiro programado a la vida útil"]

def _F(t, a, b): return 1 - np.exp(-(np.maximum(t, 0) / b) ** a)

def simular_raee(total_sistemas=5000, potencia_w=450, paneles_por_sistema=3, modo=MODOS_RAEE[0], vida=25, factor_recuperacion=1.0, anio_ini=2026, anio_fin=2030, horizonte=2065):
    años = np.arange(anio_ini, horizonte + 1)
    cohortes = np.arange(anio_ini, anio_fin + 1)
    w = 1 / (1 + np.exp(-(np.linspace(-2, 2, len(cohortes)))))            # adopción logística creciente
    inst = total_sistemas * w / w.sum()
    masa_panel = potencia_w * KG_POR_W
    fin_pan = np.zeros(len(años)); fin_inv = np.zeros(len(años)); activos = np.zeros(len(años))
    for c, n_sis in zip(cohortes, inst):
        n_pan = n_sis * paneles_por_sistema
        for j, y in enumerate(años):
            edad = y - c
            if edad < 0: continue
            if modo.startswith("Retiro"):
                f = 1.0 if edad == vida else 0.0
                f_inv = _F(edad + 1, 3.0, 13.0) - _F(edad, 3.0, 13.0)
            else:
                a, b = (5.38, 30.0) if modo.startswith("Weibull") else (2.5, 18.0)
                f = _F(edad + 1, a, b) - _F(edad, a, b)
                f_inv = _F(edad + 1, 3.0, 13.0) - _F(edad, 3.0, 13.0)
            fin_pan[j] += n_pan * f; fin_inv[j] += n_sis * f_inv
    res = pd.DataFrame({"año": años, "paneles_fin_vida": fin_pan, "inversores_fin_vida": fin_inv})
    res["residuo_paneles_t"] = res.paneles_fin_vida * masa_panel / 1000.0
    res["residuo_inversores_t"] = res.inversores_fin_vida * MASA_INVERSOR_KG / 1000.0
    res["residuo_total_t"] = res.residuo_paneles_t + res.residuo_inversores_t
    for mat, frac in COMPOSICION.items():
        res[f"{mat}_recuperable_t"] = res.residuo_paneles_t * frac * min(1.0, RECUPERACION[mat] * factor_recuperacion)
    res["recuperable_total_t"] = res[[c for c in res.columns if c.endswith("_recuperable_t")]].sum(axis=1)
    res["a_disposicion_final_t"] = res.residuo_paneles_t - res["recuperable_total_t"] + res.residuo_inversores_t
    return res

def fig_raee(res, titulo):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=res["año"], y=res["recuperable_total_t"], name="Materiales recuperables (t)", marker_color="#1e8449", secondary_y=False)
    fig.add_bar(x=res["año"], y=res["a_disposicion_final_t"], name="A disposición/valorización (t)", marker_color="#c0392b", secondary_y=False)
    fig.add_scatter(x=res["año"], y=res["residuo_total_t"].cumsum(), name="RAEE acumulado (t)", mode="lines", line=dict(color="black"), secondary_y=True)
    fig.update_layout(barmode="stack", title=titulo, height=500, xaxis_title="Año"); fig.update_yaxes(title_text="t por año", secondary_y=False); fig.update_yaxes(title_text="t acumuladas", secondary_y=True)
    return fig

def ejecutar_raee(total, pot, npan, modo, vida, rec_pct):
    r = simular_raee(int(total), float(pot), int(npan), modo, int(vida), rec_pct / 100.0)
    pico = r.loc[r.residuo_total_t.idxmax()]
    mats = {m: r[f"{m}_recuperable_t"].sum() for m in COMPOSICION}
    md = (f"### Simulación RAEE fotovoltaico 2026-2065\n- Instalaciones: **{int(total):,} sistemas** × {int(npan)} paneles de {int(pot)} W (2026-2030)  |  masa por panel ≈ **{pot*KG_POR_W:.1f} kg**\n"
          f"- RAEE total generado hasta 2065: **{r.residuo_total_t.sum():,.1f} t**  (paneles {r.residuo_paneles_t.sum():,.1f} t + inversores {r.residuo_inversores_t.sum():,.1f} t)\n"
          f"- Año pico: **{int(pico['año'])}** con **{pico.residuo_total_t:,.1f} t**\n- Materiales recuperables: **{r.recuperable_total_t.sum():,.1f} t** "
          f"({100*r.recuperable_total_t.sum()/max(r.residuo_total_t.sum(),1e-9):.0f}% del RAEE) → " + ", ".join(f"{k.split(' (')[0]} {v:,.1f} t" for k, v in mats.items() if v > 0) +
          "\n\n*Supuestos: composición típica c-Si, tasas de recuperación y curvas Weibull ilustrativas; verificar con IEA-PVPS/IRENA (2016) y con la reglamentación colombiana de RAEE (Ley 1672 de 2013).*")
    tab = r[r.residuo_total_t > 0.0005][["año", "paneles_fin_vida", "inversores_fin_vida", "residuo_total_t", "recuperable_total_t", "a_disposicion_final_t"]].round(3)
    if tab.empty: tab = r[["año", "paneles_fin_vida", "residuo_total_t"]].round(3)
    return md, fig_raee(r, f"RAEE FV por año - {modo}"), tab

for i, m in enumerate(MODOS_RAEE):
    r_ = simular_raee(5000, 450, 3, m)
    reg_fig(f"raee_escenario_{i+1}", fig_raee(r_, f"RAEE - {m}"), clave=(i == 2)); reg_tabla(f"raee_escenario_{i+1}", r_.round(4))
# ciclo de vida de un sistema típico (25 años): producción acumulada vs. huella de residuo
kwp_t = float(df["generacion_potencial_kwp"].mean()); gen_anual = float(df["generacion_mensual_estimada_kwh"].mean() * 12)
años_v = np.arange(0, VIDA_UTIL + 1)
gen_acum = np.concatenate([[0], np.cumsum(gen_anual * (1 - DEGRAD) ** np.arange(VIDA_UTIL))])
fig_vida = make_subplots(specs=[[{"secondary_y": True}]])
fig_vida.add_scatter(x=años_v, y=gen_acum / 1000, name="Generación acumulada (MWh)", mode="lines", secondary_y=False)
fig_vida.add_scatter(x=años_v, y=gen_acum * FE_CO2_KG_KWH / 1000, name="CO₂ evitado acumulado (t)", mode="lines", line=dict(dash="dash"), secondary_y=True)
fig_vida.update_layout(title=f"Ciclo de vida de un sistema típico ({kwp_t:.2f} kWp, degradación {DEGRAD*100:.1f}%/año)", height=420, xaxis_title="Año de operación")
reg_fig("raee_ciclo_vida", fig_vida)
log("RAEE completado")

# =============================================================================
# SIMULADORES
# =============================================================================
def _mediana(c): return float(df[c].median())
def _moda(c): return df[c].mode().iloc[0]
OPC = {c: sorted(df[c].dropna().unique().tolist()) for c in CAT_CLS + ["orientacion_cubierta"]}

def predecir_adopcion(estrato, zona, personas, menores, mayores, ingreso, consumo, tarifa, area_viv, area_cub, n_pan, p_pan, pct_ahorro, retorno,
                      educ, ocup, empresa, tcub, sombra, internet, smart, medidor, subsidio, reciclaje, comunidad):
    if menores + mayores >= personas:
        return {"Error": 1.0}, go.Figure(), "⚠️ Menores + adultos mayores debe ser menor que el total de personas."
    fila = pd.DataFrame([{"estrato": int(estrato), "zona": zona, "num_personas": personas, "num_menores": menores, "num_adultos_mayores": mayores,
                          "ingreso_mensual_cop": ingreso, "consumo_kwh_mes": consumo, "tarifa_kwh_cop": tarifa, "area_vivienda_m2": area_viv, "area_cubierta_m2": area_cub,
                          "num_paneles_estimados": n_pan, "potencia_panel_w": p_pan, "porcentaje_ahorro": pct_ahorro, "retorno_simple_anios": retorno,
                          "nivel_educativo": educ, "ocupacion_principal": ocup, "tiene_empresa": empresa, "tipo_cubierta": tcub, "sombras_presentes": sombra,
                          "tiene_internet": internet, "tiene_smartphone": smart, "tiene_medidor_inteligente": medidor, "recibe_subsidio": subsidio,
                          "conoce_reciclaje_paneles": reciclaje, "interes_comunidad_energetica": comunidad}])
    fila = ingenieria_caracteristicas(fila)[FEATS_CLS]
    p = MODELO_FINAL_CLF.predict_proba(fila)[0]
    dic = {CLASES[i]: float(p[i]) for i in range(3)}
    fig = go.Figure(go.Bar(x=CLASES, y=[dic[c] * 100 for c in CLASES], marker_color=[COLOR_T[c] for c in CLASES], text=[f"{dic[c]*100:.1f}%" for c in CLASES], textposition="auto"))
    fig.update_layout(title="Probabilidades de adopción", yaxis_title="%", yaxis_range=[0, 100], height=380)
    mejor = max(dic, key=dic.get)
    consejo = {"Sí": "Hogar candidato prioritario: ofrecer visita técnica y esquema de pago.", "Tal vez": "Hogar indeciso: trabajar con simulación de ahorro, capacitación y garantía de confianza.",
               "No": "Probable rechazo: priorizar subsidios/financiación y reducir la percepción de costo antes de insistir."}[mejor]
    return dic, fig, f"**Clase más probable: {mejor}** ({dic[mejor]*100:.1f}%). {consejo}\n\n*Modelo: {MEJOR}{' + calibración isotónica' if USAR_CAL else ''}. Resultado orientativo (datos sintéticos).*"

def predecir_ahorro(estrato, zona, personas, consumo, tarifa, area_cub, inclin, orient, tcub, sombra, n_pan, p_pan, ratio_dia):
    kwp = n_pan * p_pan / 1000.0; area_req = n_pan * p_pan / 205.0
    aviso = "" if area_req <= 0.75 * area_cub else f"\n\n⚠️ El área requerida ({area_req:.1f} m²) supera el 75% de la cubierta: revisar viabilidad física."
    fila = pd.DataFrame([{"estrato": int(estrato), "num_personas": personas, "consumo_kwh_mes": consumo, "tarifa_kwh_cop": tarifa, "area_cubierta_m2": area_cub,
                          "inclinacion_cubierta_grados": inclin, "num_paneles_estimados": n_pan, "potencia_panel_w": p_pan, "ratio_diurno_6_17": ratio_dia / 100.0,
                          "zona": zona, "tipo_cubierta": tcub, "orientacion_cubierta": orient, "sombras_presentes": sombra}])[FEATS_REG]
    ahorro = float(REG["ahorro_anual_cop"]["modelo"].predict(fila)[0]); ret = float(REG["retorno_simple_anios"]["modelo"].predict(fila)[0])
    factura_anual = consumo * tarifa * 12; ahorro = float(np.clip(ahorro, 0, 0.8 * factura_anual)); inv = kwp * COSTO_KWP_REF
    fig = go.Figure(go.Bar(x=["Factura anual actual", "Ahorro anual estimado", "Factura con FV"], y=[factura_anual, ahorro, factura_anual - ahorro],
                           marker_color=["#7f8c8d", "#1e8449", "#2471a3"], text=[f"${v/1e6:.2f} M" for v in (factura_anual, ahorro, factura_anual - ahorro)], textposition="auto"))
    fig.update_layout(title="Impacto económico anual (COP)", height=380)
    md = (f"### Resultados\n- Sistema: **{kwp:.2f} kWp** ({n_pan} × {p_pan} W) | área requerida **{area_req:.1f} m²**\n- Ahorro anual predicho: **${ahorro:,.0f} COP** ({100*ahorro/factura_anual:.0f}% de la factura)\n"
          f"- Retorno simple predicho: **{ret:.1f} años** (modelo directo) | referencial inversión/ahorro: **{inv/max(ahorro,1):.1f} años**\n- Inversión referencial: **${inv:,.0f} COP** (costo mediano {COSTO_KWP_REF/1e6:.2f} M COP/kWp)\n"
          f"- Modelos: ahorro → *{REG['ahorro_anual_cop']['mejor']}*, retorno → *{REG['retorno_simple_anios']['mejor']}*{aviso}")
    return md, fig

# =============================================================================
# BLOQUE 10. EXPORTACIÓN FINAL (se ejecuta antes de levantar el dashboard para poder ofrecer las descargas)
# =============================================================================
log("BLOQUE 10: Exportación de modelos, tablas y ZIP")
def guardar_modelo(obj, nombre):
    ruta = f"{DIRS['modelos']}/{nombre}.joblib"
    try:
        joblib.dump(obj, ruta, compress=3); MODELOS[nombre] = ruta
    except Exception as e:
        print("  (no se pudo guardar", nombre, ":", str(e)[:80], ")")

for n, p in PIPES.items(): guardar_modelo(p, "clf_" + re.sub(r"[^A-Za-z0-9]+", "_", n).strip("_"))
guardar_modelo(MODELO_FINAL_CLF, "clf_modelo_final_simulador")
for tgt in TARGETS_REG: guardar_modelo(REG[tgt]["modelo"], f"reg_{tgt}")
guardar_modelo({"kmeans": KM, "scaler": sc_cl, "pca2d": pca_cl, "variables": CL_FEATS, "nombres_segmentos": NOMBRES_CL}, "clustering_kmeans")
guardar_modelo({"isolation_forest": IFO, "variables": list(feat_an.columns)}, "anomalias_isolation_forest")
for nm, d in TS.items():
    try:
        d["sarimax"].remove_data()
    except Exception:
        pass
    guardar_modelo(d["sarimax"], "ts_sarimax_" + ("consumo" if nm == "Consumo" else "generacion"))
    if d["prophet"] is not None:
        try:
            from prophet.serialize import model_to_json
            ruta = f"{DIRS['modelos']}/ts_prophet_{'consumo' if nm == 'Consumo' else 'generacion'}.json"
            with open(ruta, "w") as fh: fh.write(model_to_json(d["prophet"]["modelo"]))
            MODELOS["ts_prophet_" + nm] = ruta
        except Exception as e:
            print("  (Prophet no serializado:", str(e)[:60], ")")
import sklearn, plotly, statsmodels
meta = {"semilla": SEED, "clases": CLASES, "features_clasificacion": FEATS_CLS, "features_regresion": FEATS_REG, "mejor_clasificador": MEJOR, "calibrado": bool(USAR_CAL),
        "mejores_regresores": {t: REG[t]["mejor"] for t in REG}, "k_kmeans": K_OPT, "versiones": {"sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
        "plotly": plotly.__version__, "statsmodels": statsmodels.__version__, "gradio": gr.__version__}, "aviso": "Datos sintéticos. Cumplir Ley 1581/2012 con datos reales."}
with open(f"{OUT}/metadatos.json", "w", encoding="utf-8") as fh: json.dump(meta, fh, ensure_ascii=False, indent=2)
with open(f"{OUT}/LEEME.txt", "w", encoding="utf-8") as fh:
    fh.write("Proyecto Integrador - Autogeneración Solar FV en Hogares de Popayán (2026-2030)\nDATOS SINTÉTICOS.\n\ntablas/  -> resultados en CSV\ngraficos/ -> gráficos Plotly interactivos (HTML) y PNG si el entorno lo permite\n"
             "modelos/ -> modelos entrenados (.joblib; Prophet en .json). Cargar con joblib.load(); el pipeline de clasificación acepta un DataFrame con las columnas de metadatos.json\nmetadatos.json -> variables y versiones\n")
for f in glob.glob(f"{OUT}/*.zip"): os.remove(f)
ZIP_PATH = f"{OUT}/resultados_proyecto_solar_popayan.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta, _, archivos in os.walk(OUT):
        for a in archivos:
            ruta = os.path.join(carpeta, a)
            if ruta != ZIP_PATH: z.write(ruta, os.path.relpath(ruta, OUT))
print(f"ZIP creado: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB) | {len(TABLAS)} tablas | {len(FIGS)} gráficos | {len(MODELOS)} modelos")

# =============================================================================
# BLOQUE 9. SÚPER DASHBOARD GRADIO (23 pestañas)
# =============================================================================
log("BLOQUE 9: Dashboard Gradio")
def _fmt_df(d, nd=3):
    d = d.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]): d[c] = d[c].round(nd)
    return d

m_b = tab_te.loc[MEJOR]
n_si = int((df.dispuesto_instalar_paneles == "Sí").sum())
TXT_PORTADA = f"""
# ☀️ Autogeneración Solar Fotovoltaica en Hogares de Popayán, Cauca
### Una propuesta desde la Energía Circular Inteligente 5.0 y la IA (2026-2030)
**Proyecto Integrador · Diplomado en Energía Circular Inteligente 5.0 e Inteligencia Artificial**

> ⚠️ **Datos sintéticos.** El dataset ({len(df):,} hogares, {df_raw.shape[1]} variables) fue generado y calibrado con parámetros de referencia de Popayán
> (radiación ≈ 4,25 kWh/m²/día - UPME-IDEAM; estratos 1-3; tarifas 700-950 COP/kWh). No contiene personas reales. Con datos reales aplicaría la **Ley 1581 de 2012**.

**Territorio:** Popayán (≈ 2°26′N, 76°37′O; 1.760-1.800 m s. n. m.; 13-25 °C) · ~320.000 habitantes · radiación solar media 4,0-4,5 kWh/m²/día.

#### Qué contiene este tablero
| Eje | Contenido |
|---|---|
| Datos y EDA | Diccionario, calidad, univariado, bivariado, PCA/t-SNE, correlaciones, target y Pareto de barreras |
| IA supervisada | Clasificación de adopción (7 modelos, calibración, SHAP, fairness) y regresión de ahorro/retorno + 2 simuladores |
| IA no supervisada | Segmentación K-Means / jerárquica y hogares atípicos con DBSCAN |
| Series temporales | STL, ACF/PACF, SARIMAX vs Prophet (backtest 6 meses) y anomalías con Isolation Forest |
| Prescriptivo | Asignación óptima de **$1.000 M COP** de subsidios por estrato y simulador de RAEE fotovoltaico a 25+ años |

**Cifras rápidas:** {n_si} hogares dispuestos ("Sí", {100*n_si/len(df):.0f}%) · consumo medio {df.consumo_kwh_mes.mean():.0f} kWh/mes · sistema medio {df.generacion_potencial_kwp.mean():.2f} kWp ·
inversión media ${df.inversion_estimada_cop.mean()/1e6:.1f} M COP · retorno simple medio {df.retorno_simple_anios.mean():.1f} años · mejor clasificador: **{MEJOR}** (AUC macro {m_b.roc_auc_ovr:.3f}, F1-macro {m_b.f1_macro:.3f} en prueba).
"""

def fig_corr_metodo(metodo):
    c = df[NUM_EDA].corr(method=metodo.lower()).round(2)
    f = px.imshow(c, zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto", title=f"Matriz de correlación ({metodo})"); f.update_layout(height=780)
    return f

def vista_multivariada(op):
    return {"PCA 2D": fig_pca2, "PCA 3D": fig_pca3, "t-SNE": fig_tsne, "Varianza explicada (PCA)": fig_var, "Dendrograma (Ward)": fig_dend0}[op]

def pareto_ui(poblacion, nivel):
    f, t = fig_pareto(poblacion, "Barrera" if nivel.startswith("Barrera") else "Segmentos")
    return f, t

def fairness_ui(col):
    t, r = FAIR[col]; f = fig_fairness(col)
    md = "**Métricas de disparidad** (0 = paridad perfecta; impacto dispar < 0,8 se considera alerta):\n\n" + "\n".join(f"- {k}: **{v}**" for k, v in r.items())
    md += ("\n\n*Lectura:* una diferencia en la tasa predicha de 'Sí' puede reflejar diferencias reales de adopción (ver tasa_real_Si) y no sesgo del modelo; "
           "la comparación clave es la sensibilidad (TPR) y la exactitud entre grupos. El género del jefe de hogar NO es variable del modelo y se audita como atributo protegido.")
    return f, _fmt_df(t.reset_index()), md

def shap_force_ui(clase, idx): return fig_shap_force(int(idx), clase)
def shap_bee_ui(clase): return fig_shap_beeswarm(clase)
def reg_diag_ui(tgt_label):
    tgt = [k for k, v in TARGETS_REG.items() if v == tgt_label][0]
    return FIGS[f"reg_diagnostico_{tgt}"], _fmt_df(REG[tgt]["tabla"].reset_index(), 4)
def stl_ui(serie, frec): return fig_stl(serie, frec), fig_acf_pacf(serie, frec)
def pron_ui(serie): return fig_pronostico(serie)
def uni_ui(v): return fig_univariado(v)
def biv_ui(x, y, c): return fig_bivariado(x, y, c)

NUM_FORM = NUM_EDA + ["num_menores", "num_adultos_mayores", "tarifa_kwh_cop"]
def slider(lbl, col, lo, hi, step=1, escala=1.0):
    return gr.Slider(minimum=lo, maximum=hi, value=float(np.clip(df[col].median() / escala, lo, hi)), step=step, label=lbl)

with gr.Blocks(title="Solar Popayán 2026-2030 | Energía Circular Inteligente 5.0 e IA") as demo:
    gr.Markdown("# ☀️ Solar Popayán 2026-2030 · Dashboard de Energía Circular Inteligente 5.0 e IA")
    with gr.Tabs():
        # 1
        with gr.TabItem("1. Portada e Intro"):
            gr.Markdown(TXT_PORTADA)
        # 2
        with gr.TabItem("2. Diccionario de Datos"):
            gr.Markdown("Diccionario completo (incluye las variables creadas con ingeniería de características, marcadas con **[FE]**) y reglas de coherencia verificadas.")
            gr.Dataframe(value=diccionario, interactive=False, wrap=True)
            gr.Markdown("#### Verificación de coherencia interna")
            gr.Dataframe(value=calidad, interactive=False)
        # 3
        with gr.TabItem("3. EDA Univariado"):
            d3 = gr.Dropdown(choices=NUM_EDA + CAT_EDA, value="consumo_kwh_mes", label="Variable")
            p3 = gr.Plot(value=fig_univariado("consumo_kwh_mes")); d3.change(uni_ui, d3, p3)
            gr.Markdown("#### Estadísticas descriptivas"); gr.Dataframe(value=_fmt_df(desc_num.reset_index().rename(columns={"index": "variable"})), interactive=False)
        # 4
        with gr.TabItem("4. EDA Bivariado"):
            with gr.Row():
                x4 = gr.Dropdown(choices=NUM_EDA + CAT_EDA, value="ingreso_mensual_cop", label="Variable X")
                y4 = gr.Dropdown(choices=NUM_EDA + CAT_EDA, value="consumo_kwh_mes", label="Variable Y")
                c4 = gr.Dropdown(choices=["(ninguno)"] + CAT_EDA, value="estrato", label="Color (solo numérico vs numérico)")
            b4 = gr.Button("Graficar", variant="primary"); p4 = gr.Plot(value=FIGS["eda_bivariado_consumo_ingreso"]); b4.click(biv_ui, [x4, y4, c4], p4)
        # 5
        with gr.TabItem("5. EDA Multivariado"):
            r5 = gr.Radio(["PCA 2D", "PCA 3D", "t-SNE", "Varianza explicada (PCA)", "Dendrograma (Ward)"], value="PCA 2D", label="Vista")
            p5 = gr.Plot(value=fig_pca2); r5.change(vista_multivariada, r5, p5)
            gr.Markdown("#### Cargas de los 3 primeros componentes"); gr.Dataframe(value=cargas.reset_index().rename(columns={"index": "variable"}), interactive=False)
        # 6
        with gr.TabItem("6. Matriz de Correlación"):
            r6 = gr.Radio(["Spearman", "Pearson"], value="Spearman", label="Método"); p6 = gr.Plot(value=fig_corr); r6.change(fig_corr_metodo, r6, p6)
        # 7
        with gr.TabItem("7. Análisis del Target"):
            gr.Markdown(f"**Balance del target:** " + " · ".join(f"{k}: {v:.0f}%" for k, v in bal.items()) + ". Las clases 'Tal vez' y 'Sí' conforman el 60% con interés.")
            gr.Plot(value=FIGS["analisis_target"])
        # 8
        with gr.TabItem("8. Análisis de Pareto"):
            gr.Markdown("Regla 80/20 aplicada a la **principal barrera de adopción**: ¿qué pocas causas explican la mayor parte del problema?")
            with gr.Row():
                r8a = gr.Radio(["No + Tal vez", "Solo No", "Todos"], value="No + Tal vez", label="Población")
                r8b = gr.Radio(["Barrera", "Segmentos (estrato | zona | barrera)"], value="Barrera", label="Nivel de análisis")
            p8 = gr.Plot(value=fig_par); t8 = gr.Dataframe(value=tabla_par, interactive=False)
            r8a.change(pareto_ui, [r8a, r8b], [p8, t8]); r8b.change(pareto_ui, [r8a, r8b], [p8, t8])
        # 9
        with gr.TabItem("9. Comparación de Modelos (Clasificación)"):
            gr.Markdown(f"Siete modelos comparados con validación cruzada estratificada (5 folds) y set de prueba. **Selección por F1-macro en CV: {MEJOR}.**")
            gr.Plot(value=fig_cmp); gr.Plot(value=fig_cv)
            cols9 = ["cv_f1_macro", "cv_roc_auc_ovr", "accuracy", "balanced_acc", "f1_macro", "roc_auc_ovr", "pr_auc_macro", "mcc", "kappa", "brier", "log_loss"]
            gr.Dataframe(value=_fmt_df(tab_clf[cols9].reset_index()), interactive=False)
        # 10
        with gr.TabItem("10. Métricas del Mejor Modelo"):
            gr.Markdown(f"### {MEJOR}\n" + " · ".join(f"**{k}** = {m_b[k]:.3f}" for k in ["accuracy", "f1_macro", "roc_auc_ovr", "pr_auc_macro", "mcc", "kappa", "brier"]))
            gr.Plot(value=FIGS["clf_matriz_confusion"]); gr.Plot(value=FIGS["clf_roc_pr"])
        # 11
        with gr.TabItem("11. Calibración y Brier Score"):
            gr.Markdown("Calibración **isotónica** (CV=5) sobre el mejor modelo. Brier multiclase = media de Σ(p−y)²; ECE = error de calibración esperado.")
            gr.Plot(value=FIGS["clf_calibracion"]); gr.Dataframe(value=tab_cal.reset_index().rename(columns={"index": "métrica"}), interactive=False)
        # 12
        with gr.TabItem("12. Interpretabilidad (SHAP)"):
            gr.Markdown(f"SHAP sobre **{ARBOL}** (mejor modelo de árboles) agregado por variable original; Permutation Importance sobre el modelo final.")
            gr.Plot(value=FIGS["clf_shap_resumen"])
            with gr.Row():
                c12 = gr.Dropdown(CLASES, value="Sí", label="Clase a explicar")
                i12 = gr.Slider(0, len(X_te) - 1, value=0, step=1, label="Hogar del set de prueba (force/waterfall)")
            pb = gr.Plot(value=FIGS["clf_shap_beeswarm_si"]); pf = gr.Plot(value=FIGS["clf_shap_force_ejemplo"])
            c12.change(lambda c: (shap_bee_ui(c)), c12, pb); c12.change(shap_force_ui, [c12, i12], pf); i12.change(shap_force_ui, [c12, i12], pf)
            gr.Plot(value=FIGS["clf_permutation_importance"])
        # 13
        with gr.TabItem("13. Análisis de Fairness"):
            d13 = gr.Dropdown(["estrato", "zona", "estrato_zona", "jefe_hogar_genero"], value="estrato", label="Grupo protegido / de interés")
            f0, t0, m0 = fairness_ui("estrato")
            p13 = gr.Plot(value=f0); t13 = gr.Dataframe(value=t0, interactive=False); m13 = gr.Markdown(m0)
            d13.change(fairness_ui, d13, [p13, t13, m13])
        # 14
        with gr.TabItem("14. Predicción de Adopción (Simulador 1)"):
            gr.Markdown("Ingrese las características del hogar. Los valores iniciales son las medianas/modas del dataset. Tip: obtenga *% de ahorro* y *retorno* del Simulador 2.")
            with gr.Row():
                s_est = gr.Dropdown([1, 2, 3], value=1, label="Estrato"); s_zon = gr.Dropdown(OPC["zona"], value=_moda("zona"), label="Zona")
                s_per = slider("Personas", "num_personas", 1, 8); s_men = slider("Menores", "num_menores", 0, 4); s_may = slider("Adultos mayores", "num_adultos_mayores", 0, 2)
            with gr.Row():
                s_ing = gr.Number(value=_mediana("ingreso_mensual_cop"), label="Ingreso mensual (COP)"); s_con = gr.Number(value=_mediana("consumo_kwh_mes"), label="Consumo (kWh/mes)")
                s_tar = gr.Number(value=_mediana("tarifa_kwh_cop"), label="Tarifa (COP/kWh)"); s_av = gr.Number(value=_mediana("area_vivienda_m2"), label="Área vivienda (m²)")
                s_ac = gr.Number(value=_mediana("area_cubierta_m2"), label="Área cubierta (m²)")
            with gr.Row():
                s_np = slider("Nº paneles", "num_paneles_estimados", 3, 12); s_pp = gr.Dropdown([400, 430, 450, 500, 525, 550], value=450, label="Potencia panel (W)")
                s_pa = slider("% de ahorro esperado", "porcentaje_ahorro", 30, 80, 0.5); s_re = slider("Retorno simple esperado (años)", "retorno_simple_anios", 3, 20, 0.1)
            with gr.Row():
                s_edu = gr.Dropdown(OPC["nivel_educativo"], value=_moda("nivel_educativo"), label="Nivel educativo"); s_ocu = gr.Dropdown(OPC["ocupacion_principal"], value=_moda("ocupacion_principal"), label="Ocupación")
                s_emp = gr.Dropdown(OPC["tiene_empresa"], value="No", label="Tiene empresa"); s_tc = gr.Dropdown(OPC["tipo_cubierta"], value=_moda("tipo_cubierta"), label="Tipo de cubierta")
                s_som = gr.Dropdown(OPC["sombras_presentes"], value="No", label="Sombras")
            with gr.Row():
                s_int = gr.Dropdown(OPC["tiene_internet"], value="Sí", label="Internet"); s_sma = gr.Dropdown(OPC["tiene_smartphone"], value="Sí", label="Smartphone")
                s_med = gr.Dropdown(OPC["tiene_medidor_inteligente"], value="No", label="Medidor inteligente"); s_sub = gr.Dropdown(OPC["recibe_subsidio"], value="Sí", label="Subsidio")
                s_rec = gr.Dropdown(OPC["conoce_reciclaje_paneles"], value="No", label="Conoce reciclaje de paneles"); s_com = gr.Dropdown(OPC["interes_comunidad_energetica"], value="No", label="Interés en comunidad energética")
            b14 = gr.Button("Predecir adopción", variant="primary")
            with gr.Row():
                l14 = gr.Label(label="Probabilidades", num_top_classes=3); p14 = gr.Plot()
            m14 = gr.Markdown()
            b14.click(predecir_adopcion, [s_est, s_zon, s_per, s_men, s_may, s_ing, s_con, s_tar, s_av, s_ac, s_np, s_pp, s_pa, s_re, s_edu, s_ocu, s_emp, s_tc, s_som, s_int, s_sma, s_med, s_sub, s_rec, s_com], [l14, p14, m14])
        # 15
        with gr.TabItem("15. Predicción de Ahorro (Simulador 2)"):
            with gr.Row():
                a_est = gr.Dropdown([1, 2, 3], value=1, label="Estrato"); a_zon = gr.Dropdown(OPC["zona"], value=_moda("zona"), label="Zona"); a_per = slider("Personas", "num_personas", 1, 8)
                a_con = gr.Number(value=_mediana("consumo_kwh_mes"), label="Consumo (kWh/mes)"); a_tar = gr.Number(value=_mediana("tarifa_kwh_cop"), label="Tarifa (COP/kWh)")
            with gr.Row():
                a_ac = gr.Number(value=_mediana("area_cubierta_m2"), label="Área cubierta (m²)"); a_inc = slider("Inclinación (°)", "inclinacion_cubierta_grados", 5, 35)
                a_ori = gr.Dropdown(OPC["orientacion_cubierta"], value=_moda("orientacion_cubierta"), label="Orientación"); a_tc = gr.Dropdown(OPC["tipo_cubierta"], value=_moda("tipo_cubierta"), label="Tipo de cubierta")
                a_som = gr.Dropdown(OPC["sombras_presentes"], value="No", label="Sombras")
            with gr.Row():
                a_np = slider("Nº paneles", "num_paneles_estimados", 3, 12); a_pp = gr.Dropdown([400, 430, 450, 500, 525, 550], value=450, label="Potencia panel (W)")
                a_rd = gr.Slider(35, 70, value=round(100 * _mediana("ratio_diurno_6_17"), 1), step=0.5, label="% del consumo entre 06:00 y 18:00")
            b15 = gr.Button("Estimar ahorro y retorno", variant="primary"); m15 = gr.Markdown(); p15 = gr.Plot()
            b15.click(predecir_ahorro, [a_est, a_zon, a_per, a_con, a_tar, a_ac, a_inc, a_ori, a_tc, a_som, a_np, a_pp, a_rd], [m15, p15])
            gr.Markdown("#### Desempeño de los modelos de regresión y diagnóstico de residuos")
            r15 = gr.Dropdown(list(TARGETS_REG.values()), value=TARGETS_REG["ahorro_anual_cop"], label="Objetivo")
            p15b = gr.Plot(value=FIGS["reg_diagnostico_ahorro_anual_cop"]); t15 = gr.Dataframe(value=_fmt_df(REG["ahorro_anual_cop"]["tabla"].reset_index(), 4), interactive=False)
            r15.change(reg_diag_ui, r15, [p15b, t15])
            gr.Markdown("*Nota metodológica:* en datos sintéticos el ahorro surge de reglas físicas deterministas (generación × tarifa × autoconsumo), por eso el R² es muy alto; con datos reales cabe esperar un desempeño menor.")
        # 16
        with gr.TabItem("16. Clustering K-Means"):
            gr.Markdown(f"K-Means con **k={K_OPT}** (máx. silhouette entre 3 y 7).")
            gr.Plot(value=FIGS["kmeans_seleccion_k"]); gr.Plot(value=FIGS["kmeans_pca"])
            gr.Markdown("#### Perfil de los segmentos"); gr.Dataframe(value=_fmt_df(perfil.reset_index(), 2), interactive=False)
            gr.Plot(value=FIGS["kmeans_perfil_heatmap"]); gr.Plot(value=FIGS["kmeans_perfil_horario"])
        # 17
        with gr.TabItem("17. Clustering Jerárquico"):
            gr.Markdown(f"Ward con k={K_OPT}: silhouette **{sil_h:.3f}** · Índice de Rand ajustado vs K-Means **{ARI:.2f}**. El dendrograma es interactivo (zoom/hover).")
            gr.Plot(value=FIGS["dendrograma_clustering"])
            gr.Dataframe(value=pd.crosstab(df["segmento"], df["cluster_jerarquico"]).reset_index(), interactive=False)
        # 18
        with gr.TabItem("18. Detección de Anomalías"):
            gr.Markdown(f"### Hogares atípicos (DBSCAN)\neps={EPS_DB:.2f}, min_samples={MIN_S}: **{int((lab_db == -1).sum())}** hogares atípicos ({frac_db*100:.1f}%).")
            gr.Plot(value=FIGS["dbscan_atipicos"]); gr.Dataframe(value=_fmt_df(atip), interactive=False)
            gr.Markdown(f"### Anomalías en la serie de consumo (Isolation Forest)\nPrecisión **{pr_:.2f}** · recall **{rc_:.2f}** · F1 **{f1_:.2f}** frente a las anomalías inyectadas.")
            gr.Plot(value=FIGS["ts_anomalias"])
        # 19
        with gr.TabItem("19. Series Temporales"):
            with gr.Row():
                d19a = gr.Dropdown(list(SERIES), value="Consumo", label="Serie"); d19b = gr.Radio(["Diaria", "Mensual"], value="Diaria", label="Frecuencia")
            gr.Plot(value=FIGS["ts_series"]); p19a = gr.Plot(value=FIGS["ts_stl_consumo_diaria"]); p19b = gr.Plot(value=FIGS["ts_acf_pacf_consumo"])
            d19a.change(stl_ui, [d19a, d19b], [p19a, p19b]); d19b.change(stl_ui, [d19a, d19b], [p19a, p19b])
        # 20
        with gr.TabItem("20. Pronóstico Temporal"):
            d20 = gr.Dropdown(list(SERIES), value="Consumo", label="Serie"); p20 = gr.Plot(value=FIGS["ts_pronostico_consumo"]); d20.change(pron_ui, d20, p20)
            gr.Markdown("#### Backtest (entrenar hasta abr-2026, validar may-oct 2026; error sobre sumas mensuales)")
            gr.Dataframe(value=tab_bt, interactive=False)
            gr.Markdown("*Lectura:* con solo 24 meses de historia sintética el naive estacional es una referencia exigente; compare MAE/MAPE antes de elegir modelo.")
            gr.Dataframe(value=TABLAS["ts_pronostico_12_meses"], interactive=False)
        # 21
        with gr.TabItem("21. Optimización de Presupuesto"):
            gr.Markdown(f"Programación lineal entera ({SOLVER_USADO['nombre']}): ¿cómo repartir el presupuesto de subsidios entre estratos 1, 2 y 3? "
                        f"Hogares de Popayán ≈ {HOGARES_MUN:,}; CO₂: {FE_CO2_KG_KWH} kg/kWh (referencia, verificar con UPME). Los subsidios por estrato son parámetros de política editables.")
            with gr.Row():
                o_pre = gr.Slider(100, 5000, value=1000, step=50, label="Presupuesto (millones COP)"); o_eq = gr.Slider(0, 90, value=0, step=5, label="Mínimo % del gasto en E1 (equidad)")
            with gr.Row():
                o_s1 = gr.Slider(0, 1, value=0.80, step=0.05, label="Subsidio E1 (% inversión)"); o_s2 = gr.Slider(0, 1, value=0.50, step=0.05, label="Subsidio E2"); o_s3 = gr.Slider(0, 1, value=0.20, step=0.05, label="Subsidio E3")
            with gr.Row():
                o_we = gr.Slider(0, 1, value=0.5, step=0.05, label="Peso energía/CO₂"); o_wh = gr.Slider(0, 1, value=0.3, step=0.05, label="Peso hogares atendidos"); o_wa = gr.Slider(0, 1, value=0.2, step=0.05, label="Peso ahorro familiar")
            b21 = gr.Button("Optimizar", variant="primary"); m21 = gr.Markdown(); p21 = gr.Plot(); t21 = gr.Dataframe(interactive=False)
            ent21 = [o_pre, o_s1, o_s2, o_s3, o_eq, o_we, o_wh, o_wa]
            b21.click(ejecutar_optimizacion, ent21, [m21, p21, t21]); demo.load(ejecutar_optimizacion, ent21, [m21, p21, t21])
            gr.Markdown("#### Escenarios de referencia ($1.000 M)"); gr.Plot(value=FIGS["opt_sin_equidad"]); gr.Plot(value=FIGS["opt_con_equidad"])
            gr.Plot(value=FIGS["opt_frontera_equidad"]); gr.Plot(value=FIGS["opt_sensibilidad_presupuesto"])
        # 22
        with gr.TabItem("22. Economía Circular"):
            gr.Markdown("Simulador de **RAEE fotovoltaico** (paneles + inversores) para las instalaciones 2026-2030 con proyección hasta 2065, y potencial de recuperación de materiales.")
            with gr.Row():
                e_tot = gr.Slider(500, 50000, value=5000, step=500, label="Sistemas instalados 2026-2030"); e_pot = gr.Dropdown([400, 430, 450, 500, 525, 550], value=450, label="Potencia panel (W)")
                e_npa = gr.Slider(3, 12, value=3, step=1, label="Paneles por sistema")
            with gr.Row():
                e_mod = gr.Radio(MODOS_RAEE, value=MODOS_RAEE[0], label="Modelo de fin de vida"); e_vid = gr.Slider(20, 30, value=25, step=1, label="Vida útil (solo retiro programado)")
                e_rec = gr.Slider(50, 120, value=100, step=5, label="Eficiencia de recuperación (% de la base)")
            b22 = gr.Button("Simular RAEE", variant="primary"); m22 = gr.Markdown(); p22 = gr.Plot(); t22 = gr.Dataframe(interactive=False)
            ent22 = [e_tot, e_pot, e_npa, e_mod, e_vid, e_rec]
            b22.click(ejecutar_raee, ent22, [m22, p22, t22]); demo.load(ejecutar_raee, ent22, [m22, p22, t22])
            gr.Plot(value=FIGS["raee_ciclo_vida"])
            gr.Dataframe(value=pd.DataFrame({"material": list(COMPOSICION), "% masa del panel": [v * 100 for v in COMPOSICION.values()], "recuperación asumida %": [RECUPERACION[k] * 100 for k in COMPOSICION]}), interactive=False)
        # 23
        with gr.TabItem("23. Conclusiones y Descargas"):
            t_e = df.groupby("estrato").agg(c=("consumo_kwh_mes", "mean"), r=("retorno_simple_anios", "mean"), v=("vpn_10_anios_cop", lambda s: 100 * (s > 0).mean()), si=("dispuesto_instalar_paneles", lambda s: 100 * (s == "Sí").mean()))
            top_b = tabla_par.iloc[0]
            gr.Markdown(f"""
### Conclusiones principales (datos sintéticos)
1. **Viabilidad económica desigual:** retorno simple medio de {t_e.loc[1,'r']:.1f} / {t_e.loc[2,'r']:.1f} / {t_e.loc[3,'r']:.1f} años en estratos 1/2/3; VPN a 10 años positivo en {t_e.loc[1,'v']:.0f}% / {t_e.loc[2,'v']:.0f}% / {t_e.loc[3,'v']:.0f}% de los hogares.
   Sin subsidios o financiación, el estrato 1 difícilmente recupera la inversión dentro del horizonte 2026-2030.
2. **Barrera dominante:** "{top_b.categoria}" ({top_b.pct:.0f}% de los hogares sin adopción clara); las dos primeras barreras acumulan {tabla_par.pct_acumulado.iloc[1]:.0f}% → priorizar subsidios/financiación y capacitación.
3. **IA supervisada:** el mejor clasificador ({MEJOR}) alcanza AUC macro {m_b.roc_auc_ovr:.2f}; la clase "Tal vez" es la más difícil. El techo es modesto porque la decisión sintética incluye ruido: es un resultado honesto, no un fallo.
4. **Equidad:** la disposición a adoptar crece con el estrato ({t_e.loc[1,'si']:.0f}% → {t_e.loc[3,'si']:.0f}% de "Sí"); un criterio puramente eficiente concentraría el subsidio en el estrato 3.
   La optimización muestra el costo de exigir equidad (ver pestaña 21).
5. **Segmentación:** {K_OPT} segmentos de hogares permiten diseñar ofertas diferenciadas; {int((lab_db == -1).sum())} hogares atípicos requieren estudio caso a caso.
6. **Energía circular:** los residuos de paneles de las instalaciones 2026-2030 aparecen principalmente después de 2045-2055; hay tiempo para crear la cadena de recolección y reciclaje (Ley 1672/2013).

### Limitaciones
Datos sintéticos; series temporales simuladas (24 meses); parámetros de política, factor de emisión y curvas de fallo son supuestos a validar con fuentes oficiales (UPME, XM, CREG) antes de citar cifras como resultados reales.

### Descargas
""")
            archivos = [ZIP_PATH] + sorted(glob.glob(f"{DIRS['tablas']}/*.csv")) + sorted(glob.glob(f"{DIRS['modelos']}/*"))
            gr.File(value=archivos, label="ZIP completo, CSVs de tablas y modelos (.joblib)", file_count="multiple", interactive=False)
            gr.Markdown(f"Los gráficos interactivos (HTML) están en `{DIRS['graficos']}/` y dentro del ZIP.")

N_TABS = sum(1 for b in demo.blocks.values() if isinstance(b, gr.Tab))
print(f"Dashboard construido con {N_TABS} pestañas (gr.TabItem).")

# =============================================================================
# RESUMEN FINAL Y LANZAMIENTO
# =============================================================================
print("=" * 100); print(f"PIPELINE COMPLETO en {(time.time() - T0) / 60:.1f} min | Resultados en: {os.path.abspath(OUT)}"); print("ZIP:", os.path.abspath(ZIP_PATH)); print("=" * 100)
if LAUNCH:
    try:
        demo.queue().launch(share=True, debug=False, show_error=True)    # en Colab imprime un enlace público temporal (72 h)
    except Exception as _e:
        print("No se pudo abrir el enlace público (", str(_e)[:80], "). Abriendo en modo local/inline.")
        demo.queue().launch(share=False, debug=False, show_error=True)

Instalando paquetes faltantes: ['pulp<4', 'kaleido']
AVISO ÉTICO Y DE DATOS: el dataset utilizado es SINTÉTICO, generado y calibrado con parámetros de referencia
de Popayán (UPME-IDEAM, DANE). No contiene información personal de personas reales. Aun así, cualquier uso con
datos reales de hogares debe cumplir la Ley 1581 de 2012 (Habeas Data) y el Decreto 1377 de 2013:
consentimiento informado, finalidad explícita, minimización de datos y anonimización.
[   53.1s] BLOQUE 1: ETL y Feature Engineering
Dataset cargado: ./dataset_solar_popayan.csv -> 1000 filas x 99 columnas
                       regla_de_coherencia  filas_que_incumplen cumplimiento
              Sin valores nulos (por fila)                    0       100.0%
                            id_hogar único                    0       100.0%
                      Sin filas duplicadas                    0       100.0%
             Suma h00-h23 = consumo diario                    0       100.0%
                Factura = consumo x ta

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Preparación de los datos textuales (Serialización de hogares a lenguaje natural)
def serializar_hogar(row):
    texto = f"Hogar de estrato {row['estrato']} ubicado en la zona {row['zona']}. "
    texto += f"Tiene {row['num_personas']} personas con un ingreso mensual de {row['ingreso_mensual_cop']:.0f} COP. "
    texto += f"El consumo eléctrico mensual es de {row['consumo_kwh_mes']:.1f} kWh, con una tarifa de {row['tarifa_kwh_cop']:.0f} COP por kWh. "
    texto += f"La vivienda tiene un área de {row['area_vivienda_m2']:.1f} m2 con {row['area_cubierta_m2']:.1f} m2 de cubierta disponible. "
    texto += f"Nivel educativo del jefe de hogar: {row['nivel_educativo']}. Ocupación principal: {row['ocupacion_principal']}. "
    texto += f"Tiene internet: {row['tiene_internet']}. Tiene smartphone: {row['tiene_smartphone']}. "
    texto += f"El retorno estimado de la inversión solar es a {row['retorno_simple_anios']:.1f} años con un ahorro del {row['porcentaje_ahorro']:.1f}%. "
    return texto

# Crear prompts narrativos
df['narrativa_hogar'] = df.apply(serializar_hogar, axis=1)

print("Ejemplo de serialización a lenguaje natural para el LLM:")
print(df['narrativa_hogar'].iloc[0])

# 2. Extracción de Embeddings usando un Encoder del Estado del Arte
print("\nCargando modelo de embeddings de lenguaje...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

print("Extrayendo embeddings para los 1000 hogares...")
embeddings = embedding_model.encode(df['narrativa_hogar'].tolist(), show_progress_bar=True, convert_to_numpy=True)

# 3. Preparación de Datasets y partición
y = df['dispuesto_instalar_paneles'].map({'No': 0, 'Tal vez': 1, 'Sí': 2}).astype(int).values

X_tr_emb, X_te_emb, y_tr_emb, y_te_emb = train_test_split(
    embeddings, y, test_size=0.20, stratify=y, random_state=SEED
)

class EmbeddingDataset(Dataset):
    def __init__(self, embs, targets):
        self.embs = torch.tensor(embs, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.embs)

    def __getitem__(self, idx):
        return self.embs[idx], self.targets[idx]

train_emb_dataset = EmbeddingDataset(X_tr_emb, y_tr_emb)
test_emb_dataset = EmbeddingDataset(X_te_emb, y_te_emb)

train_emb_loader = DataLoader(train_emb_dataset, batch_size=32, shuffle=True)
test_emb_loader = DataLoader(test_emb_dataset, batch_size=64, shuffle=False)

# 4. Definición del Clasificador sobre la representación del LLM
class LLMClassifier(nn.Module):
    def __init__(self, embedding_dim=384, num_classes=3):
        super(LLMClassifier, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.LayerNorm(128),
            nn.Mish(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.Mish(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)

llm_clf = LLMClassifier(embedding_dim=embeddings.shape[1], num_classes=3).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(llm_clf.parameters(), lr=1e-3, weight_decay=1e-3)

# 5. Entrenamiento del Modelo
print("\nIniciando entrenamiento del clasificador de embeddings LLM...")
epochs = 15
for epoch in range(1, epochs + 1):
    llm_clf.train()
    total_loss = 0.0
    for batch_embs, batch_ys in train_emb_loader:
        batch_embs, batch_ys = batch_embs.to(DEVICE), batch_ys.to(DEVICE)
        optimizer.zero_grad()
        outputs = llm_clf(batch_embs)
        loss = criterion(outputs, batch_ys)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_embs.size(0)

    total_loss /= len(train_emb_loader.dataset)

    # Validación rápida
    llm_clf.eval()
    preds, reals = [], []
    with torch.no_grad():
        for batch_embs, batch_ys in test_emb_loader:
            batch_embs = batch_embs.to(DEVICE)
            outputs = llm_clf(batch_embs)
            preds.extend(outputs.argmax(dim=1).cpu().numpy())
            reals.extend(batch_ys.numpy())

    acc = accuracy_score(reals, preds)
    f1 = f1_score(reals, preds, average='macro')
    print(f"Época {epoch:02d}/{epochs:02d} | Train Loss: {total_loss:.4f} | Val Accuracy: {acc:.2%} | Val F1-Macro: {f1:.4f}")

# Evaluación final
print("\nReporte de Clasificación con Embeddings de LLM:")
print(classification_report(reals, preds, target_names=CLASES))

Ejemplo de serialización a lenguaje natural para el LLM:
Hogar de estrato 2 ubicado en la zona Rural. Tiene 1 personas con un ingreso mensual de 1510000 COP. El consumo eléctrico mensual es de 100.0 kWh, con una tarifa de 843 COP por kWh. La vivienda tiene un área de 75.0 m2 con 55.0 m2 de cubierta disponible. Nivel educativo del jefe de hogar: Secundaria. Ocupación principal: Independiente. Tiene internet: Sí. Tiene smartphone: Sí. El retorno estimado de la inversión solar es a 7.1 años con un ahorro del 75.6%. 

Cargando modelo de embeddings de lenguaje...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extrayendo embeddings para los 1000 hogares...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Iniciando entrenamiento del clasificador de embeddings LLM...
Época 01/15 | Train Loss: 1.0731 | Val Accuracy: 40.00% | Val F1-Macro: 0.1905
Época 02/15 | Train Loss: 1.0534 | Val Accuracy: 50.00% | Val F1-Macro: 0.3634
Época 03/15 | Train Loss: 1.0556 | Val Accuracy: 53.00% | Val F1-Macro: 0.3917
Época 04/15 | Train Loss: 1.0380 | Val Accuracy: 47.00% | Val F1-Macro: 0.3070
Época 05/15 | Train Loss: 1.0303 | Val Accuracy: 54.00% | Val F1-Macro: 0.3987
Época 06/15 | Train Loss: 0.9757 | Val Accuracy: 54.00% | Val F1-Macro: 0.4002
Época 07/15 | Train Loss: 0.9704 | Val Accuracy: 55.50% | Val F1-Macro: 0.4121
Época 08/15 | Train Loss: 0.9855 | Val Accuracy: 56.00% | Val F1-Macro: 0.4161
Época 09/15 | Train Loss: 0.9644 | Val Accuracy: 58.00% | Val F1-Macro: 0.4303
Época 10/15 | Train Loss: 0.9547 | Val Accuracy: 57.00% | Val F1-Macro: 0.4220
Época 11/15 | Train Loss: 0.9368 | Val Accuracy: 56.50% | Val F1-Macro: 0.4194
Época 12/15 | Train Loss: 0.9554 | Val Accuracy: 56.00% | Val F1-Mac

### 🛠️ Configuración de Túnel ngrok para Gradio
Ejecuta esta celda para instalar `pyngrok`, autenticar tu cuenta y levantar el túnel seguro hacia la interfaz de Gradio.

In [ ]:
# 1. Instalar pyngrok
!pip install -q pyngrok

from pyngrok import ngrok
from google.colab import userdata
import os

# Puerto por defecto donde Gradio levanta su servidor local
PORT = 7860

# 2. Intentar obtener el token de autenticación de ngrok de los secretos de Colab
try:
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    print("✅ Autenticación de ngrok configurada correctamente.")

    # 3. Configurar dominio personalizado si está disponible
    try:
        custom_domain = userdata.get('NGROK_DOMAIN')
        print(f"📢 Usando dominio personalizado: {custom_domain}")
        public_url = ngrok.connect(PORT, pyngrok_config=None, name=None, proto="http", domain=custom_domain)
    except Exception:
        print("ℹ️ No se detectó 'NGROK_DOMAIN' o no es válido. Creando un túnel con URL aleatoria de ngrok...")
        public_url = ngrok.connect(PORT)

    print("=" * 80)
    print(f"🚀 TÚNEL NGROK ACTIVO: {public_url}")
    print(f"Asegúrate de que tu interfaz de Gradio esté corriendo en el puerto local {PORT}")
    print("=" * 80)

except Exception as e:
    print("=" * 80)
    print("❌ ERROR DE CONFIGURACIÓN DE CREDENCIALES:")
    print("Por favor, ve al menú izquierdo de Google Colab (icono de llave 🔑 Secrets)")
    print("y agrega tu clave 'NGROK_AUTHTOKEN'. Asegúrate de permitir el acceso del Notebook.")
    print("=" * 80)

❌ ERROR DE CONFIGURACIÓN DE CREDENCIALES:
Por favor, ve al menú izquierdo de Google Colab (icono de llave 🔑 Secrets)
y agrega tu clave 'NGROK_AUTHTOKEN'. Asegúrate de permitir el acceso del Notebook.
